In [ ]:
# รันเซลล์นี้บน Colab เท่านั้น: ในเครื่องใช้ venv เดิมและไม่ติดตั้งเพิ่ม
import sys, subprocess
REQUIREMENTS = 'numpy==2.4.6\npandas==3.0.6\nscikit-learn==1.9.1\nscipy==1.17.1\nthreadpoolctl==3.7.0\nlightgbm==4.7.0\ntorch==2.14.1\nmlflow==3.17.1\nfolktables==0.0.12\nmatplotlib==3.11.2\npsutil==7.2.2\npytest==9.1.1\nunsupervised-bias-detection==0.2.6\ngradio==6.30.0\njoblib==1.6.0\nkmodes==0.12.2\n'
all_pins = REQUIREMENTS.split()
hbac_pins = [p for p in all_pins if p.startswith('unsupervised-bias-detection==')]
pins = [p for p in all_pins if p not in hbac_pins]
# ใช้ wheel CPU เพื่อลดขนาดดาวน์โหลดและไม่ติดตั้ง CUDA ที่ไม่จำเป็น
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--index-url', 'https://download.pytorch.org/whl/cpu', *[p for p in pins if p.startswith('torch==')]])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', *pins])
# HBAC 0.2.6 ระบุ numpy<2 ใน metadata แต่ทดสอบกับ env numpy2 แล้ว; ติดตั้ง dependency kmodes ใน pins ด้านบน
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-deps', *hbac_pins])


# ต้นแบบนำร่อง Fair Hidden Groups
เลือก Runtime → Run all บน CPU; ไม่ต้องอัปโหลด src หรือข้อมูลก่อน
HMDA GA 2023; ประชากรใช้ประเมินและ oracle เท่านั้น · MLflow SQLite ไม่มี artifact รายรัน


In [ ]:
# ปล่อยว่างได้: notebook จะสร้างโค้ด src ให้ครบเอง
REPO_URL = "https://github.com/ronnachaii/fair-credit-hidden-groups.git"
QUICK = True
from pathlib import Path
import subprocess, os
BASE = Path('/content/rsu_fair_pilot') if Path('/content').exists() else Path.cwd() / 'colab_pilot'
if REPO_URL and not BASE.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(BASE)])
BASE.mkdir(parents=True, exist_ok=True)
os.chdir(BASE)
for folder in ['src/fairgroups', 'tests', 'results']:
    Path(folder).mkdir(parents=True, exist_ok=True)
# ไฟล์ embedded ด้านล่างเป็นเวอร์ชันเดียวกับชุดส่งงาน แม้เลือก clone repo


In [ ]:
%%writefile requirements.txt
numpy==2.4.6
pandas==3.0.6
scikit-learn==1.9.1
scipy==1.17.1
threadpoolctl==3.7.0
lightgbm==4.7.0
torch==2.14.1
mlflow==3.17.1
folktables==0.0.12
matplotlib==3.11.2
psutil==7.2.2
pytest==9.1.1
unsupervised-bias-detection==0.2.6
gradio==6.30.0
joblib==1.6.0
kmodes==0.12.2


In [ ]:
%%writefile pytest.ini
[pytest]
testpaths = tests
addopts = --basetemp=.pytest_tmp


In [ ]:
%%writefile config.py
"""ค่าทดลองชุดเดียวกันสำหรับเครื่องส่วนตัวและ Colab CPU"""
from dataclasses import asdict, dataclass
import sys


@dataclass
class Config:
    quick: bool = True
    dataset: str = "hmda"
    state: str = "GA"
    lei_top_n: int = 30
    jtt_grid: tuple = (1.0, 2.0, 4.0)
    seeds: tuple = (17, 29, 43)
    label_fractions: tuple = (1.0,)
    ks: tuple = (2, 4, 8)
    max_rows: int = 20000
    epochs: int = 10
    scarf_epochs: int = 8
    batch_size: int = 256
    hidden: int = 64
    embedding: int = 32
    learning_rate: float = 0.002
    weight_decay: float = 0.001
    dro_eta: float = 0.05
    jtt_upweight: float = 10.0
    jtt_identification_epochs: int = 3
    corruption_rate: float = 0.6
    temperature: float = 0.5
    oof_folds: int = 5
    trees: int = 100
    min_category_frequency: int = 20
    min_fair_group: int = 30
    # macOS env นี้มี libomp หลายชุด ใช้หนึ่งเธรดเพื่อเลี่ยงการชนกัน
    threads: int = 1 if sys.platform == "darwin" else 2

    def to_dict(self):
        return asdict(self)


def make_config(quick=True):
    """โหมดเต็มใช้ข้อมูลทั้งหมดและเพิ่มจำนวนรอบฝึก"""
    cfg = Config(quick=quick)
    if not quick:
        cfg.seeds = (17, 29, 43, 59, 71)
        cfg.label_fractions = (1.0,)
        cfg.max_rows = 0
        cfg.epochs = 40
        cfg.scarf_epochs = 30
        cfg.hidden = 128
        cfg.trees = 300
    return cfg


In [ ]:
%%writefile run_pilot.py
"""จุดเริ่มต้นสำหรับ command line และ notebook"""
import argparse
import sys
import os
from pathlib import Path

if sys.platform == "darwin":
    os.environ["OMP_NUM_THREADS"] = "1"
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT / "src"))
from config import make_config
from fairgroups.run import run_experiment
from threadpoolctl import threadpool_limits


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="HMDA hidden-group fairness pilot (CPU)")
    mode = parser.add_mutually_exclusive_group()
    mode.add_argument("--quick", action="store_true", help="3 seeds, full labels, 20,000 rows")
    mode.add_argument("--full", action="store_true", help="5 seeds, full labels, all rows")
    parser.add_argument("--dataset", choices=["hmda", "acs"], default="hmda")
    parser.add_argument("--state", default="GA")
    args = parser.parse_args()
    config = make_config(quick=not args.full)
    config.dataset, config.state = args.dataset, args.state.upper()
    if args.dataset == "acs":
        from fairgroups.acs_run import run_experiment
    with threadpool_limits(limits=config.threads):
        run_experiment(config, ROOT)


In [ ]:
%%writefile src/fairgroups/__init__.py
"""ต้นแบบค้นหากลุ่มและเรียนรู้ให้ทนต่อกลุ่มเสียเปรียบ"""


In [ ]:
%%writefile src/fairgroups/acs_run.py
"""รัน pilot ทุกวิธีและเก็บหลักฐานแยกข้อมูลฝึก/ทดสอบ"""
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.preprocessing import StandardScaler
from .data import (load_data, split_data, labeled_indices, TrainPreprocessor,
                   reference_groups, categorical_frames)
from .models import set_seed, train_mlp, predict_mlp, validation_scores
from .representations import train_scarf, encode_scarf, crossfit_error
from .clustering import discover, cross_label_groups
from .efficiency import ResourceMonitor, network_cost, inference_seconds, environment
from .metrics import evaluate
from .tracking import setup_tracking, log_row
from .reporting import generate_reports


def save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding="utf-8")


def record_decision(rows, model, predict, selected, base, test, config, env, output):
    """test ใช้วัดผลเมื่อเลือก checkpoint ด้วย validation เสร็จแล้ว"""
    selection, epoch, score = selected
    row = dict(base, selection=selection, selected_epoch=epoch, validation_selection_score=score)
    row.update(evaluate(test["y"], predict(test["x"]), test["s"], config.min_fair_group))
    row["inference_seconds_per_1000"] = inference_seconds(predict, test["x"])
    if base["method"] == "ERM-LGBM":
        trees = model.booster_.dump_model(num_iteration=epoch)["tree_info"]
        row.update(parameters=np.nan, flops_per_row=np.nan,
                   tree_count=len(trees), leaf_count=sum(t["num_leaves"] for t in trees))
    else:
        row.update(network_cost(model), tree_count=np.nan, leaf_count=np.nan)
    row["mlflow_run_id"] = log_row(row, env["libraries"], config)
    rows.append(row)
    pd.DataFrame(rows).to_csv(output / "pilot_results.csv", index=False)


def run_experiment(config, root="."):
    """ต้นทางทั้งหมดเป็น relative path จึงใช้เหมือนกันใน Colab"""
    root = Path(root).resolve()
    output = root / "results" / "acs_optional"
    output.mkdir(parents=True, exist_ok=True)
    start = time.perf_counter()
    set_seed(config.seeds[0], config.threads)
    env = environment(config)
    save_json(output / "environment.json", env)
    setup_tracking(root)
    with ResourceMonitor() as overall:
        x, y, sensitive = load_data(root / "data")
        env["eligible_source_rows"] = len(y)
        rows, cluster_rows, resource_rows, audits = [], [], [], []
        for seed in config.seeds:
            print(f"seed={seed}: split / preprocess / SCARF", flush=True)
            set_seed(seed, config.threads)
            splits = split_data(x, y, sensitive, seed, config.max_rows)
            train, val, test = (splits[name] for name in ["train", "val", "test"])
            audit = {"seed": seed, "train_ids": train["ids"].tolist(),
                     "val_ids": val["ids"].tolist(), "test_ids": test["ids"].tolist(),
                     "features": list(train["x"].columns), "fits": [], "error_crossfits": []}
            def add_fit(stage, ids, **extra):
                assert set(ids).issubset(set(train["ids"]))
                assert set(ids).isdisjoint(test["ids"])
                audit["fits"].append({"stage": stage, "fit_ids": np.asarray(ids).tolist(), **extra})
            with ResourceMonitor() as pre_resource:
                processor = TrainPreprocessor(config.min_category_frequency).fit(train["x"])
                transformed = {name: processor.transform(part["x"]) for name, part in splits.items()}
                categorical = categorical_frames(splits)
            add_fit("onehot_scaler", processor.fit_ids_)
            add_fit("lgbm_categories", train["ids"])
            with ResourceMonitor() as scarf_resource:
                scarf = train_scarf(train["x"], processor, config, seed)
                scaler = StandardScaler()
                reps = {"raw": transformed["train"],
                        "scarf": scaler.fit_transform(encode_scarf(scarf, transformed["train"])).astype(np.float32)}
                # val/test ไม่เคยเข้า fit แม้ในการเรียนแบบไม่มีป้ายกำกับ
                scarf_holdout = {name: scaler.transform(encode_scarf(scarf, transformed[name]))
                                 for name in ["val", "test"]}
            add_fit("scarf", scarf.fit_ids_)
            add_fit("scarf_scaler", train["ids"])
            encoder_cost = network_cost(scarf.encoder)
            pretraining_cost = network_cost(scarf)
            resource_rows.append(dict(representation="scarf", discovery_label_fraction=0, seed=seed,
                discovery_train_seconds=scarf_resource.seconds, peak_memory_mb=scarf_resource.peak_mb,
                inference_seconds_per_1000=inference_seconds(lambda xx: encode_scarf(scarf, xx), transformed["test"]),
                **encoder_cost, pretraining_parameters=pretraining_cost["parameters"],
                pretraining_flops_per_row=pretraining_cost["flops_per_row"]))
            resource_rows.append(dict(representation="raw", discovery_label_fraction=0, seed=seed,
                discovery_train_seconds=pre_resource.seconds, peak_memory_mb=pre_resource.peak_mb,
                inference_seconds_per_1000=inference_seconds(processor.transform, test["x"]),
                parameters=0, flops_per_row=0, pretraining_parameters=0, pretraining_flops_per_row=0))
            discoveries = {}
            for rep_name, rep in reps.items():
                for k in config.ks:
                    with ResourceMonitor() as cluster_resource:
                        clusters, scores, km = discover(rep, train["ids"], train["s"], k, seed)
                        # ใช้เพียง transform/predict บน holdout เพื่อยืนยันขั้นนำไปใช้
                        for name in ["val", "test"]:
                            km.predict(transformed[name] if rep_name == "raw" else scarf_holdout[name])
                    add_fit(f"kmeans_{rep_name}_{k}", km.fit_ids_)
                    discovery_seconds = cluster_resource.seconds + (scarf_resource.seconds if rep_name == "scarf" else 0)
                    discovery_peak = max(cluster_resource.peak_mb, scarf_resource.peak_mb if rep_name == "scarf" else pre_resource.peak_mb)
                    discoveries[(rep_name, k)] = (clusters, scores, discovery_seconds, discovery_peak)
                    cluster_row = dict(representation=rep_name, k=k, seed=seed,
                        discovery_label_fraction=0, n_discovery=len(train["y"]), **scores,
                        discovery_train_seconds=discovery_seconds, peak_memory_mb=discovery_peak)
                    cluster_row["mlflow_run_id"] = log_row(cluster_row, env["libraries"], config, "clustering")
                    cluster_rows.append(cluster_row)
            vg = reference_groups(val["s"])["intersection"]
            for fraction in config.label_fractions:
                labeled = labeled_indices(train["y"], fraction, seed)
                xl, yl = transformed["train"][labeled], train["y"][labeled]
                ids = train["ids"][labeled]
                print(f"seed={seed}, labels={fraction:g}, n={len(yl)}: OOF + 16 methods", flush=True)
                with ResourceMonitor() as error_resource:
                    error_rep, error_audit = crossfit_error(xl, yl, ids, config, seed)
                error_audit["label_fraction"] = fraction
                audit["error_crossfits"].append(error_audit)
                add_fit(f"error_oof_and_scaler_{fraction}", ids)
                resource_rows.append(dict(representation="error", discovery_label_fraction=fraction, seed=seed,
                    discovery_train_seconds=error_resource.seconds, peak_memory_mb=error_resource.peak_mb,
                    inference_seconds_per_1000=np.nan,
                    parameters=np.nan, flops_per_row=np.nan, pretraining_parameters=np.nan,
                    pretraining_flops_per_row=np.nan))
                current_discoveries = dict(discoveries)
                for k in config.ks:
                    with ResourceMonitor() as er_cluster:
                        clusters, scores, km = discover(error_rep, ids, train["s"].iloc[labeled], k, seed)
                    add_fit(f"kmeans_error_{k}_{fraction}", km.fit_ids_)
                    seconds = error_resource.seconds + er_cluster.seconds
                    peak = max(error_resource.peak_mb, er_cluster.peak_mb)
                    current_discoveries[("error", k)] = (clusters, scores, seconds, peak)
                    cluster_row = dict(representation="error", k=k, seed=seed,
                        discovery_label_fraction=fraction, n_discovery=len(yl), **scores,
                        discovery_train_seconds=seconds, peak_memory_mb=peak)
                    cluster_row["mlflow_run_id"] = log_row(cluster_row, env["libraries"], config, "clustering")
                    cluster_rows.append(cluster_row)
                common = dict(seed=seed, label_fraction=fraction, n_labeled=len(yl), n_train=len(train["y"]),
                              n_val=len(val["y"]), n_test=len(test["y"]), input_dimension=xl.shape[1],
                              preprocessing_seconds=pre_resource.seconds,
                              threshold=0.5, label_budget_includes_discovery=True)
                # LightGBM ใช้ categorical โดยตรง เลือกจำนวนต้นไม้จากกริดบน validation
                with ResourceMonitor() as lgb_resource:
                    lgb = LGBMClassifier(n_estimators=config.trees, num_leaves=15, max_depth=5,
                        learning_rate=0.05, verbosity=-1, n_jobs=config.threads, random_state=seed,
                        deterministic=True, force_col_wise=True)
                    lgb.fit(categorical["train"].iloc[labeled], yl)
                    candidates = sorted(set([10, 25, 50, config.trees]))
                    scores_by_tree = {t: validation_scores(val["y"],
                        lgb.predict_proba(categorical["val"], num_iteration=t)[:, 1], vg) for t in candidates}
                base = dict(common, method="ERM-LGBM", representation="none", k=0,
                            decision_train_seconds=lgb_resource.seconds, discovery_train_seconds=0,
                            train_seconds=pre_resource.seconds + lgb_resource.seconds,
                            peak_memory_mb=max(pre_resource.peak_mb, lgb_resource.peak_mb))
                for selection in ["average", "worst_group"]:
                    epoch = max(candidates, key=lambda t: scores_by_tree[t][selection])
                    predict = lambda xx, epoch=epoch: lgb.predict_proba(xx, num_iteration=epoch)[:, 1]
                    record_decision(rows, lgb, predict, (selection, epoch, scores_by_tree[epoch][selection]),
                        base, dict(test, x=categorical["test"]), config, env, output)
                # JTT รอบระบุตัวอย่างยากมีจำนวน epoch คงที่ ไม่ใช้กลุ่มจริงเลือก
                with ResourceMonitor() as jtt_resource:
                    _, identification = train_mlp(xl, yl, transformed["val"], val["y"], vg,
                        config, seed, epochs=config.jtt_identification_epochs)
                    mistakes = (predict_mlp(identification, xl) >= 0.5) != yl
                    jtt_weights = np.where(mistakes, config.jtt_upweight, 1.0)
                methods = [("ERM-MLP", "none", 0, None, None, {}, 0, 0),
                           ("JTT", "none", 0, None, jtt_weights, {}, jtt_resource.seconds, jtt_resource.peak_mb)]
                for k in config.ks:
                    # ใช้ random k แต่ละค่าเพื่อให้จำนวนกลุ่มตรงกับ ablation ทุก k
                    random_clusters = np.random.default_rng(seed + k).integers(k, size=len(yl))
                    methods.append((f"GDRO-random-k{k}", "random", k,
                                    cross_label_groups(random_clusters, yl), None, {}, 0, 0))
                for (rep_name, k), (clusters, scores, seconds, peak) in current_discoveries.items():
                    selected_clusters = clusters if rep_name == "error" else clusters[labeled]
                    methods.append((f"GDRO-{rep_name}-k{k}", rep_name, k,
                        cross_label_groups(selected_clusters, yl), None, scores, seconds, peak))
                _, oracle_groups = np.unique(reference_groups(train["s"].iloc[labeled])["intersection"], return_inverse=True)
                methods.append(("GDRO-oracle", "oracle", 4, oracle_groups, None, {}, 0, 0))
                for method, rep_name, k, groups, weights, alignment, discovery_seconds, discovery_peak in methods:
                    with ResourceMonitor() as fit_resource:
                        checkpoints, _ = train_mlp(xl, yl, transformed["val"], val["y"], vg,
                                                   config, seed, groups=groups, sample_weights=weights)
                    decision_seconds = fit_resource.seconds + (jtt_resource.seconds if method == "JTT" else 0)
                    # JTT รอบแรกนับใน decision time แล้ว ไม่บวกซ้ำใน discovery
                    actual_discovery = 0 if method == "JTT" else discovery_seconds
                    base = dict(common, method=method, representation=rep_name, k=k, **alignment,
                        decision_train_seconds=decision_seconds, discovery_train_seconds=actual_discovery,
                        train_seconds=pre_resource.seconds + actual_discovery + decision_seconds,
                        peak_memory_mb=max(pre_resource.peak_mb, fit_resource.peak_mb, discovery_peak),
                        n_dro_groups=0 if groups is None else len(np.unique(groups)))
                    for selection, (model, epoch, score) in checkpoints.items():
                        record_decision(rows, model, lambda xx, model=model: predict_mlp(model, xx),
                            (selection, epoch, score), base, dict(test, x=transformed["test"]), config, env, output)
                pd.DataFrame(cluster_rows).to_csv(output / "clustering_results.csv", index=False)
                pd.DataFrame(resource_rows).to_csv(output / "representation_resources.csv", index=False)
            audits.append(audit)
            save_json(output / "leakage_audit.json", audits)
            for resource in resource_rows:
                if resource["seed"] == seed:
                    resource["mlflow_run_id"] = log_row(resource, env["libraries"], config, "representation")
            pd.DataFrame(resource_rows).to_csv(output / "representation_resources.csv", index=False)
        generate_reports(output)
    env.update(wall_seconds=time.perf_counter() - start, peak_process_rss_mb=overall.peak_mb,
               decision_rows=len(rows), clustering_rows=len(cluster_rows),
               completed=True, colab_runtime_verified=False)
    save_json(output / "environment.json", env)
    print(f"DONE: {len(rows)} decision rows, {len(cluster_rows)} clustering rows, {env['wall_seconds']:.2f} seconds", flush=True)
    return env


In [ ]:
%%writefile src/fairgroups/clustering.py
"""จัดกลุ่มและวัดความตรงกับกลุ่มอ้างอิงหลัง fit เท่านั้น"""
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             silhouette_score, davies_bouldin_score)
from .data import reference_groups


def discover(representation, ids, sensitive, k, seed):
    """ข้อมูลประชากรใช้คำนวณ ARI/NMI เท่านั้น ไม่ส่งให้ k-means"""
    model = KMeans(n_clusters=k, n_init=10, random_state=seed)
    clusters = model.fit_predict(representation)
    model.fit_ids_ = np.asarray(ids).copy()
    scores = {"silhouette": float(silhouette_score(representation, clusters,
               sample_size=min(1000, len(clusters)), random_state=seed)),
              "davies_bouldin": float(davies_bouldin_score(representation, clusters)),
              "reconstruction_error": np.nan}
    for name, values in reference_groups(sensitive).items():
        scores[f"ari_{name}"] = adjusted_rand_score(values, clusters)
        scores[f"nmi_{name}"] = normalized_mutual_info_score(values, clusters)
    return clusters, scores, model


def cross_label_groups(clusters, y):
    """DRO เห็น cluster × y เฉพาะแถวที่มีป้ายกำกับ"""
    _, contiguous = np.unique(np.column_stack([clusters, y]), axis=0, return_inverse=True)
    return contiguous


In [ ]:
%%writefile src/fairgroups/data.py
"""โหลด ACS และแยกข้อมูลประชากรออกก่อนแปลงตัวแปร"""
from pathlib import Path
import numpy as np
import pandas as pd
from folktables import ACSDataSource, ACSIncome
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SENSITIVE = {"SEX", "RAC1P"}
CATEGORICAL = ["COW", "SCHL", "MAR", "OCCP", "POBP", "RELP"]
NUMERIC = ["AGEP", "WKHP"]
FEATURES = NUMERIC + CATEGORICAL


def load_data(root="data"):
    """ใช้แคชเดิมก่อน ถ้าไม่มีจึงให้ folktables ดาวน์โหลด"""
    cache = Path(root) / "2018/1-Year/psam_p06.csv"
    source = ACSDataSource(survey_year="2018", horizon="1-Year",
                           survey="person", root_dir=str(root))
    frame = source.get_data(states=["CA"], download=not cache.exists())
    # ใช้กติกาคัดคนของ ACSIncome โดยเก็บเลขแถวต้นฉบับไว้ตรวจข้อมูลรั่ว
    filtered = ACSIncome._preprocess(frame)
    x = filtered[FEATURES].fillna(-1).copy()
    y = (filtered["PINCP"].to_numpy() > 50000).astype(np.int64)
    sensitive = filtered[["SEX", "RAC1P"]].astype(int).copy()
    del frame, filtered
    return x, y, sensitive


def reference_groups(sensitive):
    """กลุ่มหลักใช้ White/Non-white เพื่อลดปัญหากลุ่มเล็ก"""
    sex = sensitive.SEX.to_numpy().astype(str)
    race = np.where(sensitive.RAC1P.to_numpy() == 1, "White", "Non-white")
    return {"sex": sex, "race": race,
            "intersection": np.char.add(np.char.add(sex, "_"), race),
            "race_detail": sensitive.RAC1P.to_numpy().astype(str)}


def split_data(x, y, sensitive, seed, max_rows=0):
    """สุ่มนำร่องและแบ่ง 60/20/20 ด้วย y ร่วมกับเพศและเชื้อชาติแบบสองกลุ่ม"""
    positions = np.arange(len(y))
    strata = np.char.add(reference_groups(sensitive)["intersection"], y.astype(str))
    if max_rows and len(y) > max_rows:
        positions, _ = train_test_split(positions, train_size=max_rows,
                                       stratify=strata, random_state=seed)
    train, holdout = train_test_split(positions, test_size=0.4,
                                     stratify=strata[positions], random_state=seed)
    val, test = train_test_split(holdout, test_size=0.5,
                                stratify=strata[holdout], random_state=seed)
    return {name: {"x": x.iloc[pos].copy(), "y": y[pos],
                   "s": sensitive.iloc[pos].copy(), "ids": x.index.to_numpy()[pos]}
            for name, pos in [("train", train), ("val", val), ("test", test)]}


def labeled_indices(y, fraction, seed):
    """เลือกป้ายกำกับด้วย y เท่านั้น ไม่ต้องรู้ข้อมูลประชากร"""
    ids = np.arange(len(y))
    if fraction == 1:
        return ids
    chosen, _ = train_test_split(ids, train_size=fraction, stratify=y,
                                random_state=seed)
    return np.sort(chosen)


class TrainPreprocessor:
    """รวมค่าหายากและ one-hot โดยเรียนรู้จาก train เท่านั้น"""
    def __init__(self, min_frequency=20):
        self.encoder = ColumnTransformer([
            ("numeric", StandardScaler(), NUMERIC),
            ("category", OneHotEncoder(handle_unknown="infrequent_if_exist",
             min_frequency=min_frequency, sparse_output=False, dtype=np.float32),
             CATEGORICAL)], sparse_threshold=0)

    def fit(self, frame):
        """เก็บเลขแถวและชื่อตัวแปรไว้เป็นหลักฐาน"""
        assert not SENSITIVE.intersection(frame.columns)
        self.feature_names_in_ = list(frame.columns)
        self.fit_ids_ = frame.index.to_numpy().copy()
        self.encoder.fit(frame)
        return self

    def transform(self, frame):
        return self.encoder.transform(frame).astype(np.float32)


def categorical_frames(splits):
    """LightGBM ใช้หมวดหมู่จาก train และให้ค่าใหม่เป็น missing"""
    categories = {c: sorted(splits["train"]["x"][c].unique()) for c in CATEGORICAL}
    result = {}
    for name, part in splits.items():
        frame = part["x"].copy()
        for col in CATEGORICAL:
            frame[col] = pd.Categorical(frame[col], categories=categories[col])
        result[name] = frame
    return result


In [ ]:
%%writefile src/fairgroups/demo.py
"""Gradio สำหรับวิจัย: ค่าเริ่มต้นจาก train และไม่ต้องรับข้อมูลประชากร"""
import os
import sys
if sys.platform=='darwin': os.environ['OMP_NUM_THREADS']='1'
os.environ.setdefault('GRADIO_ANALYTICS_ENABLED','False')
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from .models import predict_mlp
from .representations import encode_scarf
from .explanations import card_text

PRIMARY=['loan_amount','income','property_value','loan_to_value_ratio','debt_to_income_ratio','loan_term','loan_type','lien_status',
         'tract_to_msa_income_percentage','tract_population','total_units']


def save_bundle(output,bundle):
    cards=pd.read_csv(output/'group_explanations.csv')
    cards=cards[(cards.seed==bundle['seed']) & (cards.method==bundle['method'])]
    bundle['cards']={int(r['group']):card_text(r) for r in cards.to_dict('records')}
    bundle['inputs']=[c for c in PRIMARY if c in bundle['defaults']][:8]
    joblib.dump(bundle,output/'demo_bundle.joblib',compress=3)


def load_bundle(output='results'):
    # โหลดเฉพาะไฟล์ที่สร้างเองในโฟลเดอร์งาน
    return joblib.load(Path(output)/'demo_bundle.joblib')


def predict_application(bundle,*values):
    if len(values)!=len(bundle['inputs']): raise ValueError('จำนวนตัวแปรไม่ตรงหน้าสาธิต')
    application=dict(bundle['defaults']);application.update(dict(zip(bundle['inputs'],values)))
    for c in bundle['processor'].categorical_: application[c]=str(application[c])
    frame=bundle['guard'].transform(pd.DataFrame([application]))
    x=bundle['processor'].transform(frame)
    erm=float(predict_mlp(bundle['erm'],x)[0]); fair=float(predict_mlp(bundle['fair'],x)[0])
    rep=x if bundle['representation']=='raw' else bundle['scarf_scaler'].transform(encode_scarf(bundle['scarf'],x)).astype(np.float32)
    group=int(bundle['clusterer'].predict(rep)[0])
    result={'โอกาสอนุมัติ ERM':erm,'โอกาสอนุมัติหลังฝึก DRO':fair,'กลุ่ม':group,'วิธี':bundle['method']}
    warning='⚠️ ต้นแบบเพื่อการวิจัยเท่านั้น ความน่าจะเป็นยังไม่ผ่านการปรับเทียบ ไม่ใช้อนุมัติสินเชื่อจริง และผลทดลองอาจไม่ได้ลดอคติ'
    return result,bundle['cards'].get(group,'ยังไม่มีการ์ดกลุ่มนี้')+'\n\n'+warning


def build_interface(output='results'):
    import gradio as gr
    bundle=load_bundle(output)
    inputs=[]
    for c in bundle['inputs']:
        default=bundle['defaults'][c]
        inputs.append(gr.Number(label=c,value=float(default)) if c in bundle['processor'].numeric_ else
                      gr.Textbox(label=c,value=str(default)))
    return gr.Interface(fn=lambda *values:predict_application(bundle,*values),inputs=inputs,
                        outputs=[gr.JSON(label='ผลคาดการณ์'),gr.Markdown(label='คำอธิบายกลุ่ม')],
                        title='ต้นแบบ HMDA: ค้นหากลุ่มผู้ขอสินเชื่อ',
                        description='เพื่อการวิจัยเท่านั้น ใช้ข้อมูลหลักประมาณ 8 ตัว; ช่องอื่นใช้ค่ากลาง/ฐานนิยมจาก train ไม่ใช่ระบบอนุมัติจริง',
                        flagging_mode='never',analytics_enabled=False)


if __name__=='__main__':
    build_interface().launch(share=True)


In [ ]:
%%writefile src/fairgroups/dro.py
"""อัปเดตน้ำหนักกลุ่มแบบ exponentiated gradient ของ group DRO"""
import torch


def group_dro_loss(losses, groups, log_weights, eta):
    """เฉลี่ย loss ในแต่ละกลุ่มก่อนถ่วงน้ำหนัก กลุ่มว่างไม่ทำให้ NaN"""
    means = torch.zeros_like(log_weights)
    present = torch.zeros_like(log_weights, dtype=torch.bool)
    for group in groups.unique():
        present[group] = True
        means[group] = losses[groups == group].mean()
    with torch.no_grad():
        log_weights.add_(eta * means.detach())
        log_weights.sub_(torch.logsumexp(log_weights, dim=0))
    weights = torch.softmax(log_weights, dim=0) * present
    weights = weights / weights.sum()
    return (weights * means).sum()


In [ ]:
%%writefile src/fairgroups/efficiency.py
"""วัดทรัพยากรจริงและนับต้นทุนของชั้น Linear"""
import platform
import threading
import time
import importlib.metadata as metadata
import subprocess
from pathlib import Path
import psutil
import torch


class ResourceMonitor:
    """สุ่ม RSS ทุก 10 ms ค่านี้รวมหน่วยความจำของกระบวนการ Python"""
    def __enter__(self):
        self.process = psutil.Process()
        self.peak = self.process.memory_info().rss
        self.start = time.perf_counter()
        self.stop = threading.Event()
        self.worker = threading.Thread(target=self._sample, daemon=True)
        self.worker.start()
        return self

    def _sample(self):
        while not self.stop.wait(0.01):
            self.peak = max(self.peak, self.process.memory_info().rss)

    def __exit__(self, *args):
        self.stop.set()
        self.worker.join()
        self.peak = max(self.peak, self.process.memory_info().rss)
        self.seconds = time.perf_counter() - self.start
        self.peak_mb = self.peak / 1024**2


def network_cost(model):
    """FLOPs นับเฉพาะ Linear ตามโจทย์ ไม่รวม activation/bias"""
    return {"parameters": sum(p.numel() for p in model.parameters()),
            "flops_per_row": sum(2 * m.in_features * m.out_features
                                 for m in model.modules() if isinstance(m, torch.nn.Linear))}


def inference_seconds(predict, x):
    """อุ่นหนึ่งรอบแล้วเฉลี่ยสามรอบ คืนวินาทีต่อ 1,000 แถว"""
    sample = x[:min(1000, len(x))]
    predict(sample)
    start = time.perf_counter()
    for _ in range(3):
        predict(sample)
    return (time.perf_counter() - start) / 3 * 1000 / len(sample)


def environment(config):
    """บันทึกเวอร์ชันและทรัพยากรที่มีให้ใช้"""
    packages = ["numpy", "pandas", "scipy", "threadpoolctl", "scikit-learn", "lightgbm", "torch",
                "mlflow", "folktables", "matplotlib", "psutil", "pytest", "gradio", "unsupervised-bias-detection", "joblib"]
    cpu_model = platform.processor() or platform.machine()
    if platform.system() == "Darwin":
        try:
            cpu_model = subprocess.check_output(["sysctl", "-n", "machdep.cpu.brand_string"], text=True).strip()
        except (OSError, subprocess.CalledProcessError):
            pass
    elif Path("/proc/cpuinfo").exists():
        cpu_model = next((line.split(":", 1)[1].strip() for line in Path("/proc/cpuinfo").read_text().splitlines()
                          if line.startswith("model name")), cpu_model)
    return {"python": platform.python_version(), "platform": platform.platform(),
            "libraries": {p: metadata.version(p) for p in packages},
            "installed_packages": dict(sorted((d.metadata["Name"], d.version) for d in metadata.distributions() if d.metadata["Name"])),
            "cpu": cpu_model,
            "logical_cpus": psutil.cpu_count(),
            "ram_gb": psutil.virtual_memory().total / 1024**3,
            "gpu_available": torch.cuda.is_available(), "device_used": "cpu",
            "config": config.to_dict(), "seeds": list(config.seeds)}


In [ ]:
%%writefile src/fairgroups/explanations.py
"""กฎจากตัวแปรเดิม และทดสอบความต่างบน test ที่ไม่ใช้สร้างกลุ่ม"""
import json
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind, chi2_contingency
from sklearn.tree import DecisionTreeClassifier, _tree
from sklearn.metrics import f1_score
from .hmda import HMDAPreprocessor, NUMERIC, reference_groups


def rules(tree, names):
    """แสดงเฉพาะเส้นทางใบที่ทายว่าอยู่กลุ่มนี้ แต่ละเส้นทางไม่เกิน 3 เงื่อนไข"""
    found=[]
    def visit(node, conditions):
        if tree.tree_.feature[node] == _tree.TREE_UNDEFINED:
            if tree.classes_[np.argmax(tree.tree_.value[node][0])] == 1:
                found.append(('IF ' + ' AND '.join(conditions) if conditions else 'IF TRUE') + ' THEN อยู่กลุ่มนี้')
            return
        feature=names[tree.tree_.feature[node]].replace('numeric__','').replace('category__','')
        threshold=tree.tree_.threshold[node]
        visit(tree.tree_.children_left[node], conditions+[f'{feature} ≤ {threshold:.4g}'])
        visit(tree.tree_.children_right[node], conditions+[f'{feature} > {threshold:.4g}'])
    visit(0,[])
    return found or ['ไม่มีใบที่ทำนายว่าอยู่กลุ่มนี้ (อ่าน F1 ควบคู่)']


def describe_groups(method, k, seed, train, val, test, labels, erm_probability, config):
    processor=HMDAPreprocessor(config.min_category_frequency,config.lei_top_n,scale=False).fit(train['x'])
    xtr,xv = processor.transform(train['x']),processor.transform(val['x'])
    rows, tests=[] , []
    for group in np.unique(labels['train']):
        target=(labels['train']==group).astype(int)
        tree=DecisionTreeClassifier(max_depth=3,min_samples_leaf=max(20,len(target)//100),random_state=seed)
        tree.fit(xtr,target)
        vp=tree.predict(xv); vt=(labels['val']==group).astype(int)
        mask=labels['test']==group; yy=test['y'][mask]; pp=(erm_probability[mask]>=.5).astype(int)
        n=int(mask.sum()); positive=yy==1; negative=yy==0
        population={}
        for dimension, values in reference_groups(test['s']).items():
            if dimension=='intersection': continue
            population[dimension]={str(v):{'group':float((values[mask]==v).mean()) if n else None,
                                          'overall':float((values==v).mean())} for v in np.unique(values)}
        paths=rules(tree,processor.names())
        positive_leaves=[]
        def lengths(node,depth=0):
            if tree.tree_.feature[node]==_tree.TREE_UNDEFINED:
                if tree.classes_[np.argmax(tree.tree_.value[node][0])]==1: positive_leaves.append(depth)
            else:
                lengths(tree.tree_.children_left[node],depth+1); lengths(tree.tree_.children_right[node],depth+1)
        lengths(0)
        row={'seed':seed,'method':method,'k':k,'group':int(group),'n_train':int(target.sum()),'n_val':int(vt.sum()),'n_test':n,
             'fidelity_f1':float(f1_score(vt,vp,zero_division=0)), 'rule_length':sum(positive_leaves),
             'max_conditions_per_path':max(positive_leaves,default=0),'n_rules':len(positive_leaves),
             'coverage':float(vp.mean()),'actual_coverage_test':float(mask.mean()),
             'approval_rate':float(yy.mean()) if n else np.nan,'erm_error':float((yy!=pp).mean()) if n else np.nan,
             'fpr':float(pp[negative].mean()) if negative.any() else np.nan,
             'fnr':float((1-pp[positive]).mean()) if positive.any() else np.nan,
             'rules':json.dumps(paths,ensure_ascii=False),'population':json.dumps(population,ensure_ascii=False),
             'rule_fit_split':'train','fidelity_split':'validation','statistical_test_split':'test'}
        rows.append(row)
        for c in test['x']:
            a,b=test['x'].loc[mask,c],test['x'].loc[~mask,c]
            p=np.nan
            if c in NUMERIC:
                a,b=a.dropna(),b.dropna()
                if len(a)>1 and len(b)>1 and (a.var()>0 or b.var()>0):
                    p=float(ttest_ind(a,b,equal_var=False).pvalue)
                test_name='Welch t-test'
            else:
                table=pd.crosstab(pd.Series(mask,index=test['x'].index),test['x'][c])
                if table.shape[0]==2 and table.shape[1]>1:
                    p=float(chi2_contingency(table,correction=False)[1])
                test_name='chi-square'
            tests.append({'seed':seed,'method':method,'k':k,'group':int(group),'feature':c,'test':test_name,'p_value':p})
    return rows,tests,processor.fit_ids_


def card_text(row):
    r=row
    def fmt(v): return f'{v:.4f}' if pd.notna(v) else 'N/A'
    pop=json.loads(r['population']); composition=[]
    for dim, vals in pop.items():
        composition.append(dim+': '+', '.join(f"{v} {d['group']:.1%} (ทั้งชุด {d['overall']:.1%})" for v,d in vals.items() if d['group'] is not None))
    return '\n'.join([f"### {r['method']} · seed {r['seed']} · กลุ่ม {r['group']}",
        *['- '+v for v in json.loads(r['rules'])],
        f"- ขนาด train/validation/test: {r['n_train']}/{r['n_val']}/{r['n_test']} · coverage กฎบน validation {fmt(r['coverage'])}",
        f"- Fidelity F1 {fmt(r['fidelity_f1'])} · เงื่อนไขรวม {r['rule_length']} · สูงสุดต่อเส้นทาง {r['max_conditions_per_path']}",
        f"- อนุมัติจริง {fmt(r['approval_rate'])} · ERM error {fmt(r['erm_error'])} · FPR {fmt(r['fpr'])} · FNR {fmt(r['fnr'])}",
        '- ตัวแปรต่างชัดหลัง Bonferroni: '+r.get('significant_features',''),
        *['- องค์ประกอบ '+v for v in composition],
        '- สถิติและการทดสอบใช้ test เท่านั้น; กฎ fit train และวัด fidelity บน validation', ''])


def export_explanations(output, rows, tests):
    tests=pd.DataFrame(tests)
    # ปรับทั้ง family ของการทดสอบทุก seed/วิธี/กลุ่ม/ตัวแปรที่ส่งออก
    tests['p_bonferroni']=(tests.p_value*tests.p_value.notna().sum()).clip(upper=1)
    tests['significant']=tests.p_bonferroni<.05
    tests.to_csv(output/'group_feature_tests.csv',index=False)
    for row in rows:
        matching=tests[(tests.seed==row['seed']) & (tests.method==row['method']) & (tests.group==row['group']) & tests.significant]
        row['significant_features']=', '.join(f"{r.feature} (p_adj={r.p_bonferroni:.2g})" for r in matching.itertuples()) or 'ไม่พบ'
    frame=pd.DataFrame(rows);frame.to_csv(output/'group_explanations.csv',index=False)
    (output/'group_cards.md').write_text('# การ์ดอธิบายกลุ่ม HMDA\n\nกฎ one-hot ที่ ≤ 0.5 หมายถึงไม่มีค่านั้น; สถิติต่างกลุ่มเป็นเชิงสำรวจ ไม่ใช่เหตุและผล\n\n'+'\n'.join(card_text(r) for r in rows),encoding='utf-8')
    return rows


In [ ]:
%%writefile src/fairgroups/hmda.py
"""ข้อมูล HMDA: แยกประชากรและตัวแปรหลังตัดสินใจก่อนฝึกทุกขั้น"""
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

POST = set('action_taken purchaser_type interest_rate rate_spread hoepa_status total_loan_costs total_points_and_fees origination_charges discount_points lender_credits prepayment_penalty_term intro_rate_period negative_amortization interest_only_payment balloon_payment other_nonamortizing_features multifamily_affordable_units activity_year state_code'.split())
NUMERIC = 'loan_amount loan_to_value_ratio property_value income debt_to_income_ratio loan_term total_units tract_population ffiec_msa_md_median_family_income tract_to_msa_income_percentage tract_owner_occupied_units tract_one_to_four_family_homes tract_median_age_of_housing_units'.split()
CATEGORICAL = 'loan_type lien_status preapproval construction_method applicant_credit_score_type submission_of_application initially_payable_to_institution aus-1 conforming_loan_limit derived_loan_product_type derived_msa-md county_code lei'.split()
FEATURES = NUMERIC + CATEGORICAL
UNKNOWN = 'Unknown'


def sensitive_column(c):
    """credit_score_type ไม่ใช่คะแนนเครดิตจริง และไม่ใช่ตัวแปรประชากร"""
    return c in {'derived_race', 'derived_ethnicity', 'derived_sex'} or (
        c.startswith(('applicant_', 'co-applicant_')) and
        any(t in c for t in ('race', 'ethnicity', 'sex', 'age', 'observed')))


def prohibited(c):
    return c in POST or c.startswith('denial_reason-') or sensitive_column(c) or c == 'tract_minority_population_percent'


def missing(series):
    return series.isna() | series.astype(str).str.strip().str.lower().isin(['', 'exempt', 'na', 'n/a', 'nan', 'none'])


def dti_numeric(series):
    """แปลงช่วงเป็นค่ากึ่งกลางที่มีลำดับ ไม่เดาค่าจากผลอนุมัติ"""
    mapping = {'<20%': 10., '20%-<30%': 25., '30%-<36%': 33., '50%-60%': 55., '>60%': 65.}
    return pd.to_numeric(series.replace(mapping), errors='coerce')


def demographics(frame):
    race, eth = frame.derived_race.fillna(UNKNOWN), frame.derived_ethnicity.fillna(UNKNOWN)
    unavailable = race.str.contains('Not Available|Free Form Text Only|Unknown', case=False) | eth.str.contains('Not Available|Free Form Text Only|Unknown', case=False)
    # Hispanic รวมทุกเชื้อชาติ แม้ไม่มี race; ไม่มี ethnicity ไม่สมมุติว่า NH
    re = np.where(eth == 'Hispanic or Latino', 'Hispanic', np.where(unavailable, UNKNOWN,
         race.map({'White':'NH White', 'Black or African American':'NH Black', 'Asian':'NH Asian'}).fillna('Other')))
    sex = frame.derived_sex.where(frame.derived_sex.isin(['Male', 'Female', 'Joint']), UNKNOWN)
    age = frame.applicant_age.astype(str).where(frame.applicant_age.astype(str).isin(['<25','25-34','35-44','45-54','55-64','65-74','>74']), UNKNOWN)
    result = pd.DataFrame({'race_eth': re, 'sex': sex, 'age': age}, index=frame.index)
    result['intersection'] = np.where((result.race_eth != UNKNOWN) & (result.sex != UNKNOWN), result.race_eth + ' × ' + result.sex, UNKNOWN)
    return result


def reference_groups(s):
    return {c: s[c].to_numpy().astype(str) for c in ['race_eth','sex','age','intersection']}


def load_hmda(root, state='GA'):
    path = Path(root) / 'hmda' / f'hmda_2023_{state}.csv'
    if not path.exists():
        # urllib ติดตาม redirect ตามมาตรฐาน และเก็บแคชใช้รอบถัดไป
        import urllib.request
        path.parent.mkdir(parents=True, exist_ok=True)
        url = f'https://ffiec.cfpb.gov/v2/data-browser-api/view/csv?states={state}&years=2023&actions_taken=1,2,3'
        temp = path.with_suffix('.part')
        urllib.request.urlretrieve(url, temp)
        temp.replace(path)
    frame = pd.read_csv(path, low_memory=False)
    stages = [{'stage':'ข้อมูลต้นทาง', 'n':len(frame)}]
    filters = [('ซื้อที่อยู่อาศัย', pd.to_numeric(frame.loan_purpose, errors='coerce') == 1),
        ('อยู่เอง', frame.occupancy_type == 1), ('ไม่ใช่ธุรกิจ', frame.business_or_commercial_purpose != 1),
        ('ไม่ใช่ reverse mortgage', frame.reverse_mortgage != 1), ('ไม่ใช่ open-end', frame['open-end_line_of_credit'] != 1),
        ('1–4 หน่วย', pd.to_numeric(frame.total_units, errors='coerce').between(1,4)),
        ('Single Family', frame.derived_dwelling_category.str.contains('Single Family', na=False)),
        ('ผลอนุมัติ/ปฏิเสธที่กำหนด', frame.action_taken.isin([1,2,3]))]
    keep = np.ones(len(frame), dtype=bool)
    for label, mask in filters:
        keep &= mask.to_numpy()
        stages.append({'stage':label,'n':int(keep.sum())})
    frame = frame.loc[keep].copy()
    y = frame.action_taken.isin([1,2]).to_numpy().astype(int)
    return frame, y, demographics(frame), pd.DataFrame(stages)


def split_hmda(frame, y, s, seed, max_rows=0):
    positions = np.arange(len(y)); strata = s.race_eth.to_numpy() + y.astype(str)
    if max_rows and len(y) > max_rows:
        positions, _ = train_test_split(positions, train_size=max_rows, stratify=strata, random_state=seed)
    train, rest = train_test_split(positions, test_size=.4, stratify=strata[positions], random_state=seed)
    val, test = train_test_split(rest, test_size=.5, stratify=strata[rest], random_state=seed)
    return {name: {'frame':frame.iloc[idx].copy(), 'y':y[idx], 's':s.iloc[idx].copy(), 'ids':frame.index.to_numpy()[idx]}
            for name, idx in [('train',train),('val',val),('test',test)]}


class LeakageGuard:
    """เลือกคอลัมน์จาก train เท่านั้น ไม่ใช้ missing ของ validation/test"""
    def fit(self, frame, y):
        self.fit_ids_ = frame.index.to_numpy().copy()
        self.audit_, self.features_ = [], []
        for c in frame.columns:
            m = missing(frame[c]).to_numpy()
            rates = [float(m[y == v].mean()) for v in [0,1]]
            gap = abs(rates[1]-rates[0])
            reason = ('ข้อมูลประชากร/หลังตัดสินใจ/ตัวแทนพื้นที่ที่ห้ามใช้' if prohibited(c) else
                      'missing gap > 10 จุดเปอร์เซ็นต์' if gap > .10 else
                      'อยู่นอกรายการตัวแปรก่อนตัดสินใจที่อนุญาต' if c not in FEATURES else '')
            if not reason: self.features_.append(c)
            self.audit_.append({'column':c,'missing_denied':rates[0],'missing_approved':rates[1],
                                'missing_gap':gap,'dropped':bool(reason),'reason':reason})
        assert all(not prohibited(c) for c in self.features_)
        return self

    def transform(self, frame):
        result = frame[self.features_].copy()
        for c in result:
            if c in NUMERIC:
                result[c] = dti_numeric(result[c]) if c == 'debt_to_income_ratio' else pd.to_numeric(result[c], errors='coerce')
                result[c] = result[c].replace([np.inf,-np.inf],np.nan).astype(float)
            else:
                result[c] = result[c].where(~missing(result[c]), '__missing__').astype(str)
        return result


class HMDAPreprocessor:
    """แทนค่าว่าง รวมค่าหายาก และ top-N ผู้ให้กู้ด้วย train เท่านั้น"""
    def __init__(self, min_frequency=20, lei_top_n=30, scale=True):
        self.min_frequency, self.lei_top_n, self.scale = min_frequency, lei_top_n, scale

    def prepare(self, frame):
        frame = frame.copy()
        if 'lei' in frame:
            frame['lei'] = frame.lei.where(frame.lei.isin(self.top_lei_), '__other__')
        return frame

    def fit(self, frame):
        assert all(not prohibited(c) for c in frame.columns)
        self.fit_ids_ = frame.index.to_numpy().copy()
        self.feature_names_in_ = list(frame.columns)
        self.top_lei_ = frame.lei.value_counts().head(self.lei_top_n).index.tolist() if 'lei' in frame else []
        self.numeric_ = [c for c in frame if c in NUMERIC]
        self.categorical_ = [c for c in frame if c not in NUMERIC]
        numeric = [SimpleImputer(strategy='median', keep_empty_features=True)]
        if self.scale: numeric.append(StandardScaler())
        self.encoder = ColumnTransformer([
            ('numeric',make_pipeline(*numeric),self.numeric_),
            ('category',OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=self.min_frequency,
                                     sparse_output=False, dtype=np.float32),self.categorical_)], sparse_threshold=0)
        self.encoder.fit(self.prepare(frame))
        return self

    def transform(self, frame):
        return self.encoder.transform(self.prepare(frame)).astype(np.float32)

    def names(self):
        return self.encoder.get_feature_names_out().tolist()


def write_profile(output, stages, y, s):
    from .reporting import markdown_table
    stages.to_csv(output/'filter_counts.csv',index=False)
    rows=[]
    for name, values in reference_groups(s).items():
        for group in np.unique(values):
            mask=values==group
            rows.append({'dimension':name,'group':group,'n':int(mask.sum()),'approval_rate':float(y[mask].mean()),
                         'denial_rate':float(1-y[mask].mean()),'excluded_fairness':group==UNKNOWN})
    profile=pd.DataFrame(rows); profile.to_csv(output/'data_profile.csv',index=False)
    text=['# ข้อมูล HMDA หลังคัดกรอง','',markdown_table(stages,['stage'],['n']),'',
          'Unknown คงไว้ฝึก แต่ตัดจาก fairness เฉพาะมิติที่ไม่ทราบ และตัดกลุ่มเล็กตาม min_fair_group ใน CSV ผลทดลอง',
          '',markdown_table(profile,['dimension','group'],['n','approval_rate','denial_rate'])]
    (output/'data_profile.md').write_text('\n'.join(text),encoding='utf-8')


In [ ]:
%%writefile src/fairgroups/hmda_groups.py
"""จัดกลุ่ม train ภายในคลาส และทำนายรายใหม่โดยไม่รู้ y"""
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score, davies_bouldin_score
from sklearn.preprocessing import StandardScaler
from .hmda import reference_groups, UNKNOWN
from .models import predict_mlp
import torch


class ClassClusters:
    """k ต่อคลาส รวม 2k กลุ่ม; ตอนใช้จริงเลือกศูนย์กลางที่ใกล้สุดข้ามคลาส"""
    def fit(self, x, y, ids, k, seed):
        self.models_, self.centers_, self.labels_ = [], [], np.zeros(len(y), dtype=int)
        for label in [0,1]:
            mask = y == label
            km = KMeans(n_clusters=k,n_init=10,random_state=seed).fit(x[mask])
            self.models_.append(km); self.centers_.append(km.cluster_centers_)
            self.labels_[mask] = label*k + km.labels_
        self.centers_ = np.vstack(self.centers_).astype(np.float32)
        self.fit_ids_ = np.asarray(ids).copy()
        return self

    def predict(self, x):
        from sklearn.metrics import pairwise_distances_argmin
        return pairwise_distances_argmin(x, self.centers_)


class GeoGroups:
    """ใช้ minority percent เฉพาะ baseline นี้ ขอบ quantile มาจาก train"""
    def fit(self, values, ids, k=4):
        self.median_ = float(np.nanmedian(values))
        self.edges_ = np.unique(np.quantile(np.nan_to_num(values,nan=self.median_), np.arange(1,k)/k))
        self.fit_ids_ = np.asarray(ids).copy()
        return self
    def predict(self, values):
        return np.digitize(np.nan_to_num(values,nan=self.median_),self.edges_)


def hidden(model,x):
    model.eval()
    with torch.no_grad():
        return np.concatenate([model.layers[:-1](torch.as_tensor(x[i:i+2048])).numpy() for i in range(0,len(x),2048)])


def group_scores(x, labels, s, seed):
    out = {'n_clusters':len(np.unique(labels)), 'silhouette':np.nan, 'davies_bouldin':np.nan}
    if 1 < len(np.unique(labels)) < len(labels):
        out.update(silhouette=float(silhouette_score(x,labels,sample_size=min(1000,len(labels)),random_state=seed)),
                   davies_bouldin=float(davies_bouldin_score(x,labels)))
    for name, group in reference_groups(s).items():
        keep = group != UNKNOWN
        out[f'ari_{name}'] = adjusted_rand_score(group[keep],labels[keep])
        out[f'nmi_{name}'] = normalized_mutual_info_score(group[keep],labels[keep])
        out[f'alignment_excluded_{name}'] = int((~keep).sum())
    return out


def fit_hbac(x, error, ids, seed, k=4):
    """ใช้ API จริงของแพ็กเกจ; error สูงคือคุณภาพต่ำ จึงส่ง -error"""
    from unsupervised_bias_detection.cluster import BiasAwareHierarchicalKMeans
    model = BiasAwareHierarchicalKMeans(bahc_max_iter=k-1, bahc_min_cluster_size=max(20,len(x)//200),
                                       random_state=seed,n_init=5)
    model.fit(x, -error)
    model.fit_ids_ = np.asarray(ids).copy()
    return model


In [ ]:
%%writefile src/fairgroups/hmda_reporting.py
"""สร้างตารางผลจาก CSV เท่านั้น ไม่ใส่ค่าผลทดลองด้วยมือ"""
import os
from pathlib import Path
import pandas as pd
from .reporting import markdown_table


def generate_reports(output):
    output=Path(output); results=pd.read_csv(output/'hmda_pilot_results.csv')
    clusters=pd.read_csv(output/'clustering_results.csv'); explanations=pd.read_csv(output/'group_explanations.csv')
    resources=pd.read_csv(output/'representation_resources.csv'); jtt=pd.read_csv(output/'jtt_validation_grid.csv')
    main=results[results.selection=='average']
    baseline=results[results.method=='ERM-MLP'].set_index(['selection','seed'])
    inferred=results[results.representation.isin(['raw','scarf','GEORGE','HBAC','geoproxy'])].copy()
    inferred['eo_reduction']=inferred.apply(lambda r:baseline.loc[(r.selection,r.seed),'eo_gap']-r.eo_gap,axis=1)
    inferred['worst_group_gain']=inferred.apply(lambda r:r.worst_group_accuracy-baseline.loc[(r.selection,r.seed),'worst_group_accuracy'],axis=1)
    inferred.to_csv(output/'alignment_fairness.csv',index=False)
    correlations=[]
    for selection,part in inferred.groupby('selection'):
        for dim in ['race_eth','sex','age','intersection']:
            for metric in ['ari','nmi']:
                key=f'{metric}_{dim}'
                correlations.append({'selection':selection,'alignment':key,'n':len(part),
                                     'pearson_eo_reduction':part[key].corr(part.eo_reduction),
                                     'pearson_worst_group_gain':part[key].corr(part.worst_group_gain)})
    corr=pd.DataFrame(correlations);corr.to_csv(output/'alignment_fairness_correlations.csv',index=False)
    metrics=['accuracy','eo_gap','dp_difference','worst_group_accuracy','approval_rate_gap_race_eth','approval_rate_gap_sex','positive_rate']
    text=['# ผลนำร่อง HMDA','', 'ทุกตัวเลขในตารางอ่านจาก CSV; mean ± sample SD ข้าม seed · full labels เท่านั้น',
          'เกณฑ์ทำนาย 0.5 · EO/worst-group หลักใช้ race_eth × sex · Unknown ไม่นับ fairness; กลุ่มเล็กกว่า min_fair_group ไม่นับและบันทึกจำนวนใน CSV',
          '## ผลหลัก เลือก checkpoint ด้วย validation average accuracy','',markdown_table(main,['method'],metrics),'',
          '## วิธีที่เลือกโดยไม่ใช้ข้อมูลประชากร','',
          'เลือก k ภายใน raw/scarf/GEORGE ด้วย Silhouette train สูงสุด; เลือก raw หรือ SCARF จากคะแนนเดียวกัน ไม่ใช้ test และไม่ใช้ ARI/NMI หรือ fairness เลือก',
          'k หมายถึงต่อคลาสสำหรับ raw/scarf/GEORGE/random (รวม 2k กลุ่ม); HBAC ใช้สูงสุด 4 กลุ่มและ geoproxy ใช้ 4 quantile ที่กำหนดล่วงหน้า',
          markdown_table(main[main.selected_without_demographics],['representation','k'],metrics),'',
          'ตัวเลือกสุดท้ายของวิธีเสนอ (หนึ่งแถวต่อ seed):',
          markdown_table(main[main.proposed_selected],['seed','method'],metrics),'',
          '## การเลือกด้วย worst-group validation (ใช้ประชากรจริง; ผลเสริม)','',
          markdown_table(results[results.selection=='worst_group'],['method'],metrics),'',
          '## ความตรงกับประชากรบน test หลังเลือกเสร็จ','',
          'Silhouette/DBI ของ raw/scarf/GEORGE/HBAC วัด train; geoproxy วัด test เพื่อพรรณนาเท่านั้น ไม่ใช้เลือก',
          markdown_table(clusters,['representation','k'],['silhouette','davies_bouldin','ari_race_eth','nmi_race_eth','ari_sex','nmi_sex','ari_age','nmi_age','ari_intersection','nmi_intersection']),'',
          '## คำอธิบายกลุ่ม','',
          'ต้นไม้ลึกไม่เกิน 3 จากตัวแปรต้นเดิม; fidelity F1 และ coverage บน validation; t-test/χ² บน test อิสระจากการ fit กลุ่ม ปรับ Bonferroni ทั้ง family ทุกกลุ่ม/วิธี/seed/ตัวแปร',
          markdown_table(explanations,['method'],['fidelity_f1','rule_length','coverage','n_test','erm_error']),'',
          '## ความสัมพันธ์เชิงสำรวจกับการลดอคติเทียบ ERM-MLP','',
          markdown_table(corr,['selection','alignment'],['pearson_eo_reduction','pearson_worst_group_gain']),
          'แถว k ซ้ำและ seed ไม่เป็นอิสระทั้งหมด ค่าสหสัมพันธ์ไม่แสดงเหตุและผล','',
          '## ประสิทธิภาพระบบ','',
          markdown_table(results,['selection','method'],['train_seconds','inference_seconds_per_1000','peak_memory_mb','parameters','flops_per_row','tree_count','leaf_count']),'',
          'เวลาแต่ละวิธีรวม preprocessing/discovery ที่ใช้ร่วมกัน จึงห้ามรวมแถวเป็นเวลาทั้งชุด; FLOPs นับเฉพาะ Linear ของโมเดลตัดสินใจ; ต้นไม้รายงานจำนวนต้น/ใบแทน',
          markdown_table(resources,['representation'],['discovery_train_seconds','peak_memory_mb','parameters','flops_per_row']),'',
          '## ตรวจ JTT','',markdown_table(jtt,['seed','upweight'],['validation_accuracy','positive_rate']),
          'กริดเลือกด้วย validation average accuracy เท่านั้น; normalized weighted loss และ checkpoint เลือกก่อนประเมิน test',
          '## ข้อจำกัด','',
          '- HMDA สาธารณะไม่มีคะแนนเครดิต; applicant_credit_score_type เป็นประเภทวิธีคำนวณ ไม่ใช่คะแนน',
          '- อัตราอนุมัติสูง: accuracy/EO ต่ำอาจเกิดจากทายอนุมัติแทบทั้งหมด อ่าน positive_rate, F1, AUROC และข้อเตือนร่วมกัน',
          '- EO ของ intersection อาจไวต่อกลุ่มที่มีตัวอย่างปฏิเสธน้อย แม้ผ่านเกณฑ์ขนาดรวม 30 แถว จึงยังไม่ใช่ข้อสรุปที่แน่นอน',
          '- จัดกลุ่มภายในคลาสใช้ y บน train ตามวิธี GEORGE; ตอนใช้จริงหาศูนย์กลางใกล้สุดข้ามคลาสโดยไม่รู้ y จึงอาจสอดคล้องกับกลุ่มฝึกไม่สมบูรณ์',
          '- oracle ใช้ประชากร train เป็นเพดานเทียบ มี Unknown เป็นกลุ่มแยก ไม่ใช่วิธีที่เสนอ',
          '- การทดสอบตัวแปรเป็นเชิงสำรวจ; การแก้หลายการทดสอบไม่ได้ขจัดปัญหา cluster ที่นิยามจากตัวแปรเดียวกัน',
          '- ยังไม่ได้รัน Colab จริงหรือ FULL; QUICK ใช้ epochs น้อย ไม่รับประกันประโยชน์ด้าน fairness',
          '- Gradio ใช้ค่ากลาง/ฐานนิยม train เติมช่องที่ไม่ได้กรอก ผลเป็นต้นแบบวิจัยเท่านั้น','']
    collapsed=main[(main.method=='JTT') & ((main.positive_rate>=.995)|(main.positive_rate<=.005))]
    if len(collapsed): text.append('คำเตือน JTT ทายข้างเดียวใน test: '+', '.join(str(v) for v in collapsed.seed))
    (output/'hmda_pilot_summary.md').write_text('\n'.join(text),encoding='utf-8')
    os.environ.setdefault('MPLCONFIGDIR',str(output/'.matplotlib'))
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    figs=output/'figures';figs.mkdir(exist_ok=True)
    for selection,part in results.groupby('selection'):
        fig,ax=plt.subplots(figsize=(10,6))
        for method,g in part.groupby('method'):
            ax.errorbar(g.eo_gap.mean(),g.accuracy.mean(),xerr=g.eo_gap.std(),yerr=g.accuracy.std(),fmt='o',label=method)
        ax.set(xlabel='EO gap',ylabel='Accuracy',title=selection);ax.legend(fontsize=7,loc='center left',bbox_to_anchor=(1,0.5))
        fig.tight_layout();fig.savefig(figs/f'accuracy_eo_{selection}.png',dpi=140);plt.close(fig)
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for rep,g in clusters.groupby('representation'):
        axes[0].scatter(g.ari_intersection,g.nmi_intersection,label=rep)
    axes[0].set(xlabel='ARI intersection',ylabel='NMI intersection');axes[0].legend(fontsize=7)
    for method,g in explanations.groupby('method'):
        axes[1].scatter(g.coverage,g.fidelity_f1,label=method)
    axes[1].set(xlabel='Validation rule coverage',ylabel='Validation fidelity F1');axes[1].legend(fontsize=6)
    fig.tight_layout();fig.savefig(figs/'alignment_explanations.png',dpi=140);plt.close(fig)


In [ ]:
%%writefile src/fairgroups/metrics.py
"""ตัวชี้วัดการตัดสินใจและช่องว่างระหว่างกลุ่มจริง"""
import json
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
from .data import reference_groups as acs_groups
from .hmda import reference_groups as hmda_groups, UNKNOWN


def group_metrics(y, prediction, group, min_size=1):
    """EO = max(ช่วง TPR, ช่วง FPR); ไม่แทนอัตราที่ไม่มีตัวหารด้วยศูนย์"""
    details, tprs, fprs, positives = {}, [], [], []
    for value in np.unique(group):
        mask = group == value
        if mask.sum() < min_size:
            continue
        yy, pp = y[mask], prediction[mask]
        tpr = float(pp[yy == 1].mean()) if (yy == 1).any() else np.nan
        fpr = float(pp[yy == 0].mean()) if (yy == 0).any() else np.nan
        details[str(value)] = {"n": int(mask.sum()), "accuracy": float((yy == pp).mean()),
                              "tpr": tpr if np.isfinite(tpr) else None,
                              "fpr": fpr if np.isfinite(fpr) else None,
                              "positive_rate": float(pp.mean())}
        if np.isfinite(tpr):
            tprs.append(tpr)
        if np.isfinite(fpr):
            fprs.append(fpr)
        positives.append(float(pp.mean()))
    def gap(values):
        return max(values) - min(values) if len(values) >= 2 else np.nan
    return {"eo_gap": max(gap(tprs), gap(fprs)), "tpr_gap": gap(tprs),
            "fpr_gap": gap(fprs), "dp_difference": gap(positives),
            "worst_group_accuracy": min((v["accuracy"] for v in details.values()), default=np.nan),
            "n_eligible_groups": len(details), "group_accuracy": details}


def evaluate(y, probability, sensitive, min_detail=30):
    """คอลัมน์ fairness หลักใช้ intersection เพศ × White/Non-white"""
    prediction = (probability >= 0.5).astype(int)
    result = {"accuracy": accuracy_score(y, prediction),
              "precision": precision_score(y, prediction, zero_division=0),
              "recall": recall_score(y, prediction, zero_division=0),
              "f1": f1_score(y, prediction, zero_division=0),
              "auroc": roc_auc_score(y, probability)}
    refs = hmda_groups(sensitive) if "race_eth" in sensitive else acs_groups(sensitive)
    result["positive_rate"] = float(prediction.mean())
    for name, group in refs.items():
        eligible = group != UNKNOWN
        result[f"excluded_unknown_{name}"] = int((~eligible).sum())
        scores = group_metrics(y[eligible], prediction[eligible], group[eligible], min_detail)
        result[f"excluded_small_{name}"] = int(eligible.sum() - sum(v["n"] for v in scores["group_accuracy"].values()))
        result[f"approval_rate_gap_{name}"] = scores["dp_difference"]
        for metric, value in scores.items():
            result[f"{metric}_{name}"] = json.dumps(value, ensure_ascii=False) if isinstance(value, dict) else value
        if name == "intersection":
            for metric in ["eo_gap", "tpr_gap", "fpr_gap", "dp_difference", "worst_group_accuracy"]:
                result[metric] = scores[metric]
    return result


In [ ]:
%%writefile src/fairgroups/models.py
"""โมเดล MLP และการเลือก checkpoint ด้วย validation เท่านั้น"""
import copy
import random
import numpy as np
import torch
from torch import nn
from .dro import group_dro_loss


def set_seed(seed, threads=2):
    """ใช้ CPU และตั้ง seed เพื่อให้ทำซ้ำได้"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(threads)
    torch.use_deterministic_algorithms(True)


class MLP(nn.Module):
    """ทุกวิธีใช้ input และโครงข่ายตัดสินใจเดียวกันเพื่อเทียบตรง"""
    def __init__(self, input_dim, hidden):
        super().__init__()
        self.layers = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(),
                                    nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))

    def forward(self, x):
        return self.layers(x).squeeze(-1)


def predict_mlp(model, x):
    """ทำนายเป็นชุดเล็กเพื่อลด RAM"""
    model.eval()
    with torch.no_grad():
        return np.concatenate([model(torch.as_tensor(x[i:i+2048])).sigmoid().numpy()
                               for i in range(0, len(x), 2048)])


def validation_scores(y, probability, groups):
    """average ไม่แตะกลุ่มจริง ส่วน worst-group ใช้เพื่อการวิเคราะห์แยก"""
    correct = (probability >= 0.5) == y
    return {"average": float(correct.mean()),
            "worst_group": min(float(correct[groups == g].mean()) for g in np.unique(groups) if str(g) != "Unknown")}


def train_mlp(x, y, xv, yv, val_groups, config, seed, groups=None,
              sample_weights=None, epochs=None):
    """ฝึกจำนวนรอบคงที่และเก็บ checkpoint ดีที่สุดของทั้งสองกติกา"""
    set_seed(seed, config.threads)
    model = MLP(x.shape[1], config.hidden)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate,
                                 weight_decay=config.weight_decay)
    xt, yt = torch.as_tensor(x), torch.as_tensor(y, dtype=torch.float32)
    gt = None if groups is None else torch.as_tensor(groups, dtype=torch.long)
    wt = None if sample_weights is None else torch.as_tensor(sample_weights, dtype=torch.float32)
    log_weights = None if groups is None else torch.zeros(int(np.max(groups)) + 1)
    best_scores = {"average": -1, "worst_group": -1}
    best_states, best_epochs = {}, {}
    for epoch in range(epochs or config.epochs):
        model.train()
        order = torch.randperm(len(y))
        for ids in order.split(config.batch_size):
            optimizer.zero_grad()
            losses = nn.functional.binary_cross_entropy_with_logits(model(xt[ids]), yt[ids], reduction="none")
            if gt is not None:
                loss = group_dro_loss(losses, gt[ids], log_weights, config.dro_eta)
            elif wt is not None:
                # JTT ใช้น้ำหนักคงที่จากโมเดลรอบแรก ไม่เลือกด้วย test
                loss = (losses * wt[ids]).sum() / wt[ids].sum()
            else:
                loss = losses.mean()
            loss.backward()
            optimizer.step()
        scores = validation_scores(yv, predict_mlp(model, xv), val_groups)
        for selection, score in scores.items():
            if score > best_scores[selection]:
                best_scores[selection] = score
                best_states[selection] = copy.deepcopy(model.state_dict())
                best_epochs[selection] = epoch + 1
    result = {}
    for selection, state in best_states.items():
        selected = MLP(x.shape[1], config.hidden)
        selected.load_state_dict(state)
        result[selection] = (selected, best_epochs[selection], best_scores[selection])
    return result, model


In [ ]:
%%writefile src/fairgroups/reporting.py
"""สร้างรายงานและกราฟจาก CSV ที่บันทึกแล้วเท่านั้น"""
import os
from pathlib import Path
import numpy as np
import pandas as pd


def markdown_table(frame, groups, metrics):
    """ไม่ใช้ tabulate เพื่อลดแพ็กเกจที่ต้องติดตั้ง"""
    rows = ["| " + " | ".join(groups + metrics) + " |",
            "| " + " | ".join(["---"] * (len(groups) + len(metrics))) + " |"]
    for key, part in frame.groupby(groups, sort=True, dropna=False):
        key = key if isinstance(key, tuple) else (key,)
        cells = [str(v) for v in key]
        for metric in metrics:
            values = part[metric].dropna()
            if len(values) > 1:
                cells.append(f"{values.mean():.4f} ± {values.std(ddof=1):.4f}")
            else:
                cells.append(f"{values.mean():.4f}" if len(values) else "N/A")
        rows.append("| " + " | ".join(cells) + " |")
    return "\n".join(rows)


def generate_reports(root):
    """โหลดผลจากดิสก์ใหม่ ไม่ฝังตัวเลขผลทดลองลงในข้อความ"""
    root = Path(root)
    results = pd.read_csv(root / "pilot_results.csv")
    clusters = pd.read_csv(root / "clustering_results.csv")
    resources = pd.read_csv(root / "representation_resources.csv")
    groups = ["selection", "label_fraction", "method"]
    text = ["# ผลนำร่อง ACS CA 2018", "",
            "ค่าทุกตารางเป็น mean ± sample SD ข้าม seed และอ่านจาก CSV เท่านั้น", "",
            "selection=average เป็นผลหลัก; worst_group ใช้เพศ/เชื้อชาติจริงใน validation เพื่อเลือก checkpoint", "",
            "EO gap และ worst-group หลักใช้เพศ × White/Non-white; เกณฑ์ทำนาย 0.5 ทุกวิธี", "",
            "## ความแม่นยำและความเป็นธรรม", "",
            markdown_table(results, groups, ["accuracy", "precision", "recall", "f1", "auroc",
                           "eo_gap", "dp_difference", "worst_group_accuracy"]), "",
            "## ความตรงของกลุ่ม (วัดบนข้อมูล train ที่ใช้ค้นหากลุ่ม)", "",
            "raw/scarf ใช้ train ทั้งหมดโดยไม่ใช้ y; error ใช้ OOF เฉพาะ train ที่มีป้ายกำกับตามงบ", "",
            "discovery_label_fraction=0 หมายถึงไม่ใช้ป้ายกำกับ ไม่ใช่ไม่มีข้อมูลฝึก", "",
            markdown_table(clusters, ["representation", "discovery_label_fraction", "k"],
                           ["silhouette", "davies_bouldin", "ari_sex", "nmi_sex", "ari_race", "nmi_race",
                            "ari_intersection", "nmi_intersection"]), "",
            "Reconstruction error: N/A เพราะ SCARF ไม่มี decoder", "",
            "## ประสิทธิภาพระบบ", "",
            "train_seconds รวม preprocessing + discovery ที่นำกลับใช้ร่วมกัน + การฝึกตัวตัดสินใจของวิธีนั้น "
            "จึงห้ามนำเวลาแต่ละแถวมารวมเป็นเวลารันทั้งชุด; peak_memory_mb เป็น RSS รวมของ Python ที่สุ่มทุก 10 ms", "",
            markdown_table(results, groups, ["train_seconds", "decision_train_seconds",
                           "inference_seconds_per_1000", "peak_memory_mb", "parameters", "flops_per_row",
                           "tree_count", "leaf_count"]), "",
            "## ต้นทุนการค้นหา representation", "",
            "SCARF encoder แยกจาก projection head; FLOPs นับเฉพาะ Linear ตาม 2×in×out", "",
            markdown_table(resources, ["representation", "discovery_label_fraction"],
                           ["discovery_train_seconds", "inference_seconds_per_1000", "peak_memory_mb", "parameters", "flops_per_row",
                            "pretraining_parameters", "pretraining_flops_per_row"]), "",
            "## ความสัมพันธ์ของความตรงของกลุ่มกับผลความเป็นธรรม", "",
            "ค่าสหสัมพันธ์เชิงสำรวจจาก cluster methods เท่านั้น ยังไม่ใช่ข้อสรุปเชิงเหตุและผล", ""]
    relation_rows = []
    for (selection, fraction), part in results[results.representation.isin(["raw", "scarf", "error"]) &
                                                    results.method.str.startswith("GDRO-")].groupby(["selection", "label_fraction"]):
        for metric in ["ari_intersection", "nmi_intersection"]:
            relation_rows.append({"selection": selection, "label_fraction": fraction, "alignment": metric,
                "pearson_worst_group": part[metric].corr(part.worst_group_accuracy),
                "pearson_eo_gap": part[metric].corr(part.eo_gap), "n": len(part)})
    relation = pd.DataFrame(relation_rows)
    relation.to_csv(root / "alignment_fairness_correlations.csv", index=False)
    text += [markdown_table(relation, ["selection", "label_fraction", "alignment"],
                            ["pearson_worst_group", "pearson_eo_gap"]), "",
             "## ข้อจำกัด", "",
             "- เป็น pilot ขนาดย่อและ epochs น้อย ผลไม่ได้รับประกันว่าจะลดช่องว่างได้ทุกวิธี", 
             "- error เป็น label-assisted discovery จึงแยกจาก raw/scarf ที่ไม่ใช้ป้ายกำกับ",
             "- กลุ่มละเอียดขนาดต่ำกว่าเกณฑ์ถูกตัดออกจาก fairness แบบละเอียดเท่านั้น ดูจำนวนใน CSV",
             "- ใช้ POBP ซึ่งอาจเป็น proxy เชื้อชาติ; oracle ใช้ข้อมูลประชากร train และเป็นเพียงตัวเทียบ",
             "- การแบ่งข้อมูล stratified ใช้กลุ่มจริงเพื่อออกแบบการประเมิน; ตัวฝึกหลักไม่รับข้อมูลนั้น",
             "- ยังไม่ยืนยันเวลาจริงบน Colab; ARL และ Gradio อยู่ใน TODO", ""]
    # EO ต่ำอาจเกิดจากทายคลาสเดียว จึงตรวจจากอัตราทายบวกที่เก็บจริงใน CSV
    for (selection, fraction), part in results[results.method == "JTT"].groupby(["selection", "label_fraction"]):
        rates = []
        for raw in part.group_accuracy_intersection:
            import json
            groups = json.loads(raw)
            rates.append(sum(g["n"] * g["positive_rate"] for g in groups.values()) /
                         sum(g["n"] for g in groups.values()))
        if any(rate > 0.98 or rate < 0.02 for rate in rates):
            text.append(f"- ตรวจพบ JTT ทำนายกระจุกคลาสเดียวอย่างน้อยหนึ่ง seed: selection={selection}, "
                        f"labels={fraction:g}, อัตราทายบวกเฉลี่ย={np.mean(rates):.4f}; "
                        "EO gap ต่ำในกรณีนี้ไม่ใช่หลักฐานว่าตัดสินใจได้ดีขึ้น ต้องอ่าน accuracy/recall ควบคู่")
    text.append("")
    (root / "pilot_summary.md").write_text("\n".join(text), encoding="utf-8")
    make_figures(results, clusters, root)


def make_figures(results, clusters, root):
    """กราฟแยกกติกาเลือกโมเดลเพื่อไม่ปะปนการใช้ข้อมูลประชากร"""
    os.environ.setdefault("MPLCONFIGDIR", str(root / ".matplotlib"))
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    figure_dir = root / "figures"
    figure_dir.mkdir(exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for metric, ax in zip(["ari_intersection", "nmi_intersection"], axes):
        for (rep, frac), part in clusters.groupby(["representation", "discovery_label_fraction"]):
            values = part.groupby("k")[metric].agg(["mean", "std"])
            ax.errorbar(values.index, values["mean"], yerr=values["std"], marker="o", label=f"{rep} labels={frac:g}")
        ax.set(xlabel="k", ylabel=metric, xticks=[2, 4, 8])
        ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(figure_dir / "alignment.png", dpi=160)
    plt.close(fig)
    for selection, selected in results.groupby("selection"):
        fractions = sorted(selected.label_fraction.unique())
        # ให้ทั้ง 16 วิธีมีสีต่างกัน และใช้สัญลักษณ์ช่วยอ่านกรณีพิมพ์ขาวดำ
        method_names = sorted(selected.method.unique())
        colors = {name: plt.get_cmap("tab20")(i) for i, name in enumerate(method_names)}
        markers = {name: ["o", "s", "^", "D", "v", "P", "X", "*"][i % 8]
                   for i, name in enumerate(method_names)}
        fig, axes = plt.subplots(1, len(fractions), figsize=(7*len(fractions), 5), squeeze=False)
        for fraction, ax in zip(fractions, axes[0]):
            part = selected[selected.label_fraction == fraction]
            for method, group in part.groupby("method"):
                ax.errorbar(group.eo_gap.mean(), group.accuracy.mean(),
                            xerr=group.eo_gap.std(), yerr=group.accuracy.std(),
                            fmt=markers[method], color=colors[method], label=method)
            ax.set(xlabel="EO gap (lower is better)", ylabel="Accuracy", title=f"Labels {fraction:.0%} / {selection}")
        axes[0, -1].legend(fontsize=7, loc="center left", bbox_to_anchor=(1, 0.5))
        fig.tight_layout()
        fig.savefig(figure_dir / f"accuracy_eo_{selection}.png", dpi=160, bbox_inches="tight")
        plt.close(fig)
        fig, ax = plt.subplots(figsize=(11, 5))
        for method, part in selected.groupby("method"):
            values = part.groupby("label_fraction").worst_group_accuracy.agg(["mean", "std"])
            ax.errorbar(values.index, values["mean"], yerr=values["std"],
                        marker=markers[method], color=colors[method], label=method)
        ax.set(xlabel="Label fraction", ylabel="Worst-group accuracy", title=selection, xticks=fractions)
        ax.legend(fontsize=7, loc="center left", bbox_to_anchor=(1, 0.5))
        fig.tight_layout()
        fig.savefig(figure_dir / f"worst_group_labels_{selection}.png", dpi=160)
        plt.close(fig)


In [ ]:
%%writefile src/fairgroups/representations.py
"""SCARF ไม่มีป้ายกำกับ และ error representation จาก OOF ภายใน train"""
import numpy as np
import torch
from torch import nn
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier
from .models import set_seed


class ScarfEncoder(nn.Module):
    """สุ่มแทนค่าทีละ feature จาก marginal ของ train ก่อนทำ one-hot"""
    def __init__(self, input_dim, hidden, embedding):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(),
                                     nn.Linear(hidden, hidden), nn.ReLU(),
                                     nn.Linear(hidden, embedding))
        self.projector = nn.Sequential(nn.Linear(embedding, embedding), nn.ReLU(),
                                       nn.Linear(embedding, embedding))

    def forward(self, x):
        return self.encoder(x)


def contrastive_loss(a, b, temperature):
    """NT-Xent: คู่จริงคือแถวเดิมกับแถวที่แทนค่าบาง feature"""
    n = len(a)
    z = nn.functional.normalize(torch.cat([a, b]), dim=1)
    similarity = z @ z.T / temperature
    similarity.fill_diagonal_(-1e9)
    targets = (torch.arange(2*n) + n) % (2*n)
    return nn.functional.cross_entropy(similarity, targets)


def train_scarf(frame, processor, config, seed):
    """ไม่รับ y หรือข้อมูลประชากร และ donor มาจาก train เท่านั้น"""
    set_seed(seed, config.threads)
    x = processor.transform(frame)
    model = ScarfEncoder(x.shape[1], config.hidden, config.embedding)
    model.fit_ids_ = frame.index.to_numpy().copy()
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate,
                                 weight_decay=config.weight_decay)
    rng = np.random.default_rng(seed)
    # สร้าง corrupted view ครั้งละ epoch เพื่อให้หมวดหมู่ยังเป็นค่าที่ถูกต้อง
    for _ in range(config.scarf_epochs):
        corrupted = frame.copy()
        for col in frame.columns:
            donors = rng.integers(0, len(frame), len(frame))
            mask = rng.random(len(frame)) < config.corruption_rate
            corrupted.loc[mask, col] = frame[col].to_numpy()[donors[mask]]
        cx = processor.transform(corrupted)
        order = rng.permutation(len(x))
        model.train()
        for start in range(0, len(x), config.batch_size):
            ids = order[start:start+config.batch_size]
            if len(ids) < 2:
                continue
            optimizer.zero_grad()
            a = model.projector(model(torch.as_tensor(x[ids])))
            b = model.projector(model(torch.as_tensor(cx[ids])))
            loss = contrastive_loss(a, b, config.temperature)
            loss.backward()
            optimizer.step()
    return model


def encode_scarf(model, x):
    """แช่แข็ง encoder ตอนจัดกลุ่ม ไม่ใช้ projection head"""
    model.eval()
    with torch.no_grad():
        return np.concatenate([model(torch.as_tensor(x[i:i+2048])).numpy()
                               for i in range(0, len(x), 2048)])


def crossfit_error(x, y, ids, config, seed):
    """ใช้ป้ายกำกับเท่าที่มี และตรวจว่าแถวที่ทายไม่เคยเข้าโมเดล fold นั้น"""
    probability = np.full(len(y), np.nan)
    coverage = np.zeros(len(y), dtype=int)
    folds = []
    splitter = StratifiedKFold(n_splits=config.oof_folds, shuffle=True, random_state=seed)
    for train, held in splitter.split(x, y):
        assert set(ids[train]).isdisjoint(ids[held])
        model = LGBMClassifier(n_estimators=config.trees, num_leaves=15, max_depth=5,
                               verbosity=-1, n_jobs=config.threads, random_state=seed,
                               deterministic=True, force_col_wise=True)
        model.fit(x[train], y[train])
        probability[held] = model.predict_proba(x[held])[:, 1]
        coverage[held] += 1
        folds.append({"fit_ids": ids[train].tolist(), "heldout_ids": ids[held].tolist()})
    assert np.all(coverage == 1) and np.isfinite(probability).all()
    p = np.clip(probability, 1e-6, 1-1e-6)
    loss = -(y * np.log(p) + (1-y) * np.log(1-p))
    scaler = StandardScaler()
    representation = scaler.fit_transform(np.column_stack([loss, y, p])).astype(np.float32)
    return representation, {"fit_ids": ids.tolist(), "folds": folds,
                            "oof_coverage_min": int(coverage.min()),
                            "oof_coverage_max": int(coverage.max())}


In [ ]:
%%writefile src/fairgroups/run.py
"""HMDA pilot ต่อยอดโมดูลเดิม: train-only discovery และ full labels เท่านั้น"""
import json
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier
from .hmda import load_hmda, split_hmda, LeakageGuard, HMDAPreprocessor, write_profile
from .hmda_groups import ClassClusters, GeoGroups, hidden, fit_hbac, group_scores
from .models import set_seed, train_mlp, predict_mlp, validation_scores
from .representations import train_scarf, encode_scarf
from .efficiency import ResourceMonitor, network_cost, inference_seconds, environment
from .metrics import evaluate
from .tracking import setup_tracking, log_row
from .explanations import describe_groups, export_explanations


def save_json(path,value):
    Path(path).write_text(json.dumps(value,indent=2,ensure_ascii=False),encoding='utf-8')


def oof_errors(train,config,seed):
    """แต่ละ fold มี imputer/encoder ของตัวเอง แถว held ไม่เข้า fit แม้ขั้นแปลง"""
    y=train['y']; p=np.full(len(y),np.nan); folds=[]
    for fit,held in StratifiedKFold(config.oof_folds,shuffle=True,random_state=seed).split(train['x'],y):
        # ตัวกรอง missing gap อาศัย y จึงต้อง fit ใหม่ภายใน fold ด้วย
        fold_guard=LeakageGuard().fit(train['frame'].iloc[fit],y[fit])
        fold_train=fold_guard.transform(train['frame'].iloc[fit])
        fold_held=fold_guard.transform(train['frame'].iloc[held])
        pre=HMDAPreprocessor(config.min_category_frequency,config.lei_top_n).fit(fold_train)
        model=LGBMClassifier(n_estimators=config.trees,num_leaves=15,max_depth=5,verbosity=-1,
                             n_jobs=config.threads,random_state=seed,deterministic=True,force_col_wise=True)
        model.fit(pre.transform(fold_train),y[fit])
        p[held]=model.predict_proba(pre.transform(fold_held))[:,1]
        folds.append({'fit_ids':train['ids'][fit].tolist(),'heldout_ids':train['ids'][held].tolist(),
                      'guard_fit_ids':fold_guard.fit_ids_.tolist(),'preprocessor_fit_ids':pre.fit_ids_.tolist(),
                      'features':fold_guard.features_})
    assert np.isfinite(p).all()
    return ((p>=.5)!=y).astype(float),{'fit_ids':train['ids'].tolist(),'folds':folds}


def choose_jtt(xt,train,xv,val,vg,config,seed):
    """กริดเล็ก เลือกด้วย average accuracy; รายงาน collapse ทุก candidate"""
    _, identification=train_mlp(xt,train['y'],xv,val['y'],vg,config,seed,epochs=config.jtt_identification_epochs)
    mistakes=(predict_mlp(identification,xt)>=.5)!=train['y']
    candidates=[]; history=[]
    for weight in config.jtt_grid:
        checkpoints,_=train_mlp(xt,train['y'],xv,val['y'],vg,config,seed,
                                sample_weights=np.where(mistakes,weight,1.))
        p=predict_mlp(checkpoints['average'][0],xv); rate=float((p>=.5).mean())
        collapsed=rate<=.005 or rate>=.995
        history.append({'seed':seed,'upweight':weight,'validation_accuracy':checkpoints['average'][2],
                        'positive_rate':rate,'collapsed':collapsed,'n_mistakes':int(mistakes.sum())})
        candidates.append((checkpoints['average'][2],weight,checkpoints))
    # ใช้ accuracy เป็นกติกาหลักตรงกันทุกวิธี ไม่เลือกจาก fairness หรือ test
    score,weight,chosen=max(candidates,key=lambda t:t[0])
    selected=next(h for h in history if h['upweight']==weight)
    if selected['collapsed']: warnings.warn('JTT validation ทายข้างเดียว อ่าน accuracy และ rate ควบคู่')
    return chosen,weight,history


def run_experiment(config,root='.'):
    root=Path(root).resolve(); output=root/'results';output.mkdir(exist_ok=True)
    start=time.perf_counter(); set_seed(config.seeds[0],config.threads)
    before={str(p.relative_to(root)) for p in root.rglob('*') if p.is_file()}
    prior_env=json.loads((output/'environment.json').read_text()) if (output/'environment.json').exists() else {}
    env=environment(config); setup_tracking(root)
    rows=[];clusters=[];resources=[];audits=[];explanations=[];feature_tests=[];jtt_history=[];demo_bundle=None
    with ResourceMonitor() as overall:
        frame,y,s,stages=load_hmda(root/'data',config.state)
        write_profile(output,stages,y,s)
        env.update(eligible_source_rows=len(y),source_rows=int(stages.iloc[0]['n']))
        for seed in config.seeds:
            print(f'seed={seed}: HMDA split / audit / representation',flush=True)
            splits=split_hmda(frame,y,s,seed,config.max_rows)
            train,val,test=(splits[n] for n in ['train','val','test'])
            audit={'seed':seed,**{f'{n}_ids':p['ids'].tolist() for n,p in splits.items()},'fits':[]}
            def add_fit(stage,ids):
                assert set(ids).issubset(train['ids']) and set(ids).isdisjoint(np.r_[val['ids'],test['ids']])
                audit['fits'].append({'stage':stage,'fit_ids':np.asarray(ids).tolist()})
            with ResourceMonitor() as pr:
                guard=LeakageGuard().fit(train['frame'],train['y'])
                for part in splits.values(): part['x']=guard.transform(part['frame'])
                processor=HMDAPreprocessor(config.min_category_frequency,config.lei_top_n).fit(train['x'])
                transformed={n:processor.transform(p['x']) for n,p in splits.items()}
            xt,xv,xx=(transformed[n] for n in ['train','val','test'])
            add_fit('leakage_guard',guard.fit_ids_);add_fit('imputer_scaler_categories_top_lei',processor.fit_ids_)
            audit.update(features=guard.features_,column_audit=guard.audit_)
            # กลุ่มจริงใช้เลือกเฉพาะผล worst_group ที่รายงานแยก; average ไม่แตะข้อมูลนี้
            vg=val['s'].intersection.to_numpy()
            with ResourceMonitor() as er:
                erm_checkpoints,_=train_mlp(xt,train['y'],xv,val['y'],vg,config,seed)
            erm=erm_checkpoints['average'][0]
            with ResourceMonitor() as sr:
                scarf=train_scarf(train['x'],processor,config,seed)
                scarf_scaler=StandardScaler().fit(encode_scarf(scarf,xt))
                reps={'raw':transformed,'scarf':{n:scarf_scaler.transform(encode_scarf(scarf,x)).astype(np.float32) for n,x in transformed.items()}}
            add_fit('scarf',scarf.fit_ids_);add_fit('scarf_scaler',train['ids'])
            with ResourceMonitor() as gr:
                george_scaler=StandardScaler().fit(hidden(erm,xt))
                reps['GEORGE']={n:george_scaler.transform(hidden(erm,x)).astype(np.float32) for n,x in transformed.items()}
            add_fit('GEORGE_erm_hidden_scaler',train['ids'])
            for name,seconds,model in [('raw',pr.seconds,None),('scarf',sr.seconds,scarf.encoder),('GEORGE',er.seconds+gr.seconds,erm)]:
                resources.append({'seed':seed,'representation':name,'discovery_train_seconds':seconds,
                                  'peak_memory_mb':overall.process.memory_info().rss/1024**2,
                                  **(network_cost(model) if model is not None else {'parameters':0,'flops_per_row':0})})
            discoveries={}; selected={}
            for rep_name,rep in reps.items():
                for k in config.ks:
                    with ResourceMonitor() as cr:
                        km=ClassClusters().fit(rep['train'],train['y'],train['ids'],k,seed)
                        labels={'train':km.labels_,**{n:km.predict(rep[n]) for n in ['val','test']}}
                        # k เลือกจาก train เท่านั้น; ARI/NMI test เก็บภายหลังเพื่อประเมิน
                        scores=group_scores(rep['train'],labels['train'],train['s'],seed)
                        alignment=group_scores(rep['test'],labels['test'],test['s'],seed)
                    add_fit(f'class_kmeans_{rep_name}_{k}',km.fit_ids_)
                    seconds=cr.seconds+(sr.seconds if rep_name=='scarf' else er.seconds+gr.seconds if rep_name=='GEORGE' else 0)
                    rec={'seed':seed,'representation':rep_name,'k':k,'silhouette':scores['silhouette'],
                         'davies_bouldin':scores['davies_bouldin'],'n_clusters':scores['n_clusters'],
                         **{key:value for key,value in alignment.items() if key.startswith(('ari_','nmi_','alignment_'))},
                         'discovery_train_seconds':seconds,'assignment':'nearest center across classes without y'}
                    clusters.append(rec);discoveries[(rep_name,k)]=(km,labels,rec,seconds)
                selected[rep_name]=max(config.ks,key=lambda k:discoveries[(rep_name,k)][2]['silhouette'])
            # HBAC ใช้ OOF error ของ ERM-LGBM บน train ไม่ใช้ demographic
            with ResourceMonitor() as hr:
                error,oof=oof_errors(train,config,seed)
                hbac=fit_hbac(xt,error,train['ids'],seed,k=4)
                labels={'train':hbac.labels_,**{n:hbac.predict(transformed[n]) for n in ['val','test']}}
                hscore=group_scores(xt,labels['train'],train['s'],seed)
                alignment=group_scores(xx,labels['test'],test['s'],seed)
            audit['error_crossfits']=[oof];add_fit('HBAC_OOF',train['ids']);add_fit('HBAC',hbac.fit_ids_)
            hrec={'seed':seed,'representation':'HBAC','k':hbac.n_clusters_,'silhouette':hscore['silhouette'],
                  'davies_bouldin':hscore['davies_bouldin'],'n_clusters':hbac.n_clusters_,
                  **{a:b for a,b in alignment.items() if a.startswith(('ari_','nmi_','alignment_'))},'discovery_train_seconds':hr.seconds}
            clusters.append(hrec);discoveries[('HBAC',hbac.n_clusters_)]=(hbac,labels,hrec,hr.seconds)
            selected['HBAC']=hbac.n_clusters_
            resources.append({'seed':seed,'representation':'HBAC','discovery_train_seconds':hr.seconds,
                              'peak_memory_mb':hr.peak_mb,'parameters':np.nan,'flops_per_row':np.nan})
            with ResourceMonitor() as geo_r:
                geo_values={n:pd.to_numeric(p['frame'].tract_minority_population_percent,errors='coerce').to_numpy() for n,p in splits.items()}
                geo=GeoGroups().fit(geo_values['train'],train['ids'])
                geo_labels={n:geo.predict(v) for n,v in geo_values.items()}
                geo_score=group_scores(xx,geo_labels['test'],test['s'],seed)
            add_fit('geoproxy_quantiles',geo.fit_ids_)
            clusters.append({'seed':seed,'representation':'geoproxy','k':4,**geo_score,'discovery_train_seconds':geo_r.seconds})
            # อธิบายเฉพาะ k ที่เลือกไว้ โดย test ไม่เคยถูกใช้เลือก
            for rep_name,k in selected.items():
                _,labels,_,_=discoveries[(rep_name,k)]
                method=f'GDRO-{rep_name}-k{k}' if rep_name in ['raw','scarf'] else rep_name
                cards,tests,fit_ids=describe_groups(method,k,seed,train,val,test,labels,predict_mlp(erm,xx),config)
                explanations.extend(cards);feature_tests.extend(tests);add_fit(f'explanation_{rep_name}',fit_ids)
            main_rep,main_k=max([(r,selected[r]) for r in ['raw','scarf']],key=lambda pair:discoveries[pair][2]['silhouette'])
            def record(method,rep,k,checkpoints,seconds,peak,alignment=None,discovery_seconds=0,extra=None):
                for selection,(model,epoch,score) in checkpoints.items():
                    predict=(lambda a,model=model:predict_mlp(model,a)) if method!='ERM-LGBM' else (lambda a,epoch=epoch:model.predict_proba(a,num_iteration=epoch)[:,1])
                    row={'seed':seed,'method':method,'representation':rep,'k':k,'label_fraction':1.,'selection':selection,
                         'selected_epoch':epoch,'validation_selection_score':score,'n_train':len(xt),'n_val':len(xv),'n_test':len(xx),
                         'input_dimension':xt.shape[1],'threshold':.5,'train_seconds':pr.seconds+seconds+discovery_seconds,
                         'decision_train_seconds':seconds,'discovery_train_seconds':discovery_seconds,'peak_memory_mb':max(peak,pr.peak_mb),
                         'selected_without_demographics':(rep in selected and k==selected[rep]) or rep=='geoproxy',
                         'proposed_selected':rep==main_rep and k==main_k,**(alignment or {}),**(extra or {})}
                    row.update(evaluate(test['y'],predict(xx),test['s'],config.min_fair_group))
                    row['inference_seconds_per_1000']=inference_seconds(predict,xx)
                    if method=='ERM-LGBM':
                        trees=model.booster_.dump_model(num_iteration=epoch)['tree_info']
                        row.update(parameters=np.nan,flops_per_row=np.nan,tree_count=len(trees),leaf_count=sum(t['num_leaves'] for t in trees))
                    else: row.update(network_cost(model))
                    row['mlflow_run_id']=log_row(row,env['libraries'],config);rows.append(row)
            record('ERM-MLP','none',0,erm_checkpoints,er.seconds,er.peak_mb)
            with ResourceMonitor() as lr:
                lgb=LGBMClassifier(n_estimators=config.trees,num_leaves=15,max_depth=5,learning_rate=.05,verbosity=-1,
                    n_jobs=config.threads,random_state=seed,deterministic=True,force_col_wise=True)
                lgb.fit(xt,train['y'])
                choices={t:validation_scores(val['y'],lgb.predict_proba(xv,num_iteration=t)[:,1],vg) for t in sorted(set([10,25,50,config.trees]))}
                lc={sel:(lgb,t,choices[t][sel]) for sel in ['average','worst_group'] for t in [max(choices,key=lambda t:choices[t][sel])]}
            record('ERM-LGBM','none',0,lc,lr.seconds,lr.peak_mb)
            with ResourceMonitor() as jr:
                jc,jw,jhistory=choose_jtt(xt,train,xv,val,vg,config,seed)
            jtt_history.extend(jhistory);record('JTT','none',0,jc,jr.seconds,jr.peak_mb,extra={'jtt_upweight':jw})
            methods=[]
            for k in config.ks:
                # ควบคุมจำนวนกลุ่มและการใช้ y ให้เท่ากับวิธีที่จัด k กลุ่มต่อคลาส
                random=train['y']*k+np.random.default_rng(seed+k).integers(k,size=len(xt))
                methods.append((f'GDRO-random-k{k}','random',k,random,{},0))
            for (rep,k),(_,labels,rec,seconds) in discoveries.items():
                if rep=='GEORGE' and k!=selected[rep]: continue
                name=f'GDRO-{rep}-k{k}' if rep in ['raw','scarf'] else rep
                methods.append((name,rep,k,labels['train'],rec,seconds))
            methods.append(('GDRO-geoproxy','geoproxy',4,geo_labels['train'],geo_score,geo_r.seconds))
            _,oracle=np.unique(train['s'].intersection.to_numpy(),return_inverse=True)
            # Unknown ไม่ตีความเป็นประชากรที่ยืนยัน; ใช้เป็นกลุ่มฝึกแยกเพื่อไม่ทิ้งแถว
            methods.append(('GDRO-oracle','oracle',len(np.unique(oracle)),oracle,{},0))
            for method,rep,k,groups,alignment,seconds in methods:
                print(f'  {seed}: {method}',flush=True)
                with ResourceMonitor() as mr:
                    checkpoints,_=train_mlp(xt,train['y'],xv,val['y'],vg,config,seed,groups=groups)
                metrics={a:b for a,b in alignment.items() if a.startswith(('ari_','nmi_','silhouette','davies_bouldin','n_clusters'))}
                record(method,rep,k,checkpoints,mr.seconds,mr.peak_mb,metrics,seconds,extra={'n_dro_groups':len(np.unique(groups))})
                if seed==config.seeds[0] and rep==main_rep and k==main_k:
                    demo_bundle={'guard':guard,'processor':processor,'erm':erm,'fair':checkpoints['average'][0],
                        'representation':rep,'scarf':scarf,'scarf_scaler':scarf_scaler,'clusterer':discoveries[(rep,k)][0],
                        'defaults':{c:float(train['x'][c].median()) if pd.api.types.is_numeric_dtype(train['x'][c]) else train['x'][c].mode().iloc[0] for c in train['x']},
                        'method':method,'seed':seed}
            audits.append(audit)
            pd.DataFrame(rows).to_csv(output/'hmda_pilot_results.csv',index=False)
            pd.DataFrame(clusters).to_csv(output/'clustering_results.csv',index=False)
            save_json(output/'leakage_audit.json',{'scope':'train-only per seed','seeds':audits})
        export_explanations(output,explanations,feature_tests)
        pd.DataFrame(resources).to_csv(output/'representation_resources.csv',index=False)
        pd.DataFrame(jtt_history).to_csv(output/'jtt_validation_grid.csv',index=False)
        from .demo import save_bundle
        save_bundle(output,demo_bundle)
        from .hmda_reporting import generate_reports
        generate_reports(output)
    env.update(wall_seconds=time.perf_counter()-start,peak_process_rss_mb=overall.peak_mb,decision_rows=len(rows),
               clustering_rows=len(clusters),completed=True,colab_runtime_verified=False,hbac_backend='unsupervised-bias-detection 0.2.6')
    after={str(p.relative_to(root)) for p in root.rglob('*') if p.is_file()}
    env.update(new_files=len(after-before),total_files=len(after),new_file_limit=200,
               max_new_files_completed_runs=max(prior_env.get('max_new_files_completed_runs',prior_env.get('new_files',0)),len(after-before)))
    pd.DataFrame([{k:env[k] for k in ['wall_seconds','peak_process_rss_mb','decision_rows','clustering_rows','new_files','total_files','max_new_files_completed_runs']}]).to_csv(output/'runtime_summary.csv',index=False)
    assert env['new_files']<=200,env['new_files']
    save_json(output/'environment.json',env)
    print(f"DONE: {len(rows)} decision rows; {env['wall_seconds']:.2f} s; peak RSS {overall.peak_mb:.1f} MiB; new files {env['new_files']}",flush=True)
    return env


In [ ]:
%%writefile src/fairgroups/tracking.py
"""SQLite เก็บ params/metrics/tags โดยไม่มี artifact รายรัน"""
from pathlib import Path
import numpy as np
import mlflow


def setup_tracking(root):
    import os
    os.environ.setdefault('MLFLOW_ENABLE_TELEMETRY', 'false')
    root = Path(root).resolve()
    mlflow.set_tracking_uri('sqlite:///' + str(root / 'mlflow.db'))
    client = mlflow.tracking.MlflowClient()
    name = 'hmda-hidden-groups-pilot'
    exp = client.get_experiment_by_name(name)
    if exp is None:
        exp_id = client.create_experiment(name, artifact_location=(root/'results'/'no_run_artifacts').as_uri())
    else:
        exp_id = exp.experiment_id
    mlflow.set_experiment(experiment_id=exp_id)


def log_row(row, versions, config, kind='decision'):
    keys = ['method','representation','k','label_fraction','seed','selection','selected_epoch','n_train','n_test']
    with mlflow.start_run(run_name=f"{kind}-{row.get('method', row.get('representation','none'))}-{row['seed']}") as run:
        mlflow.set_tags({'kind':kind,'dataset':config.dataset,'state':config.state,'artifact_policy':'none'})
        mlflow.log_params({k:row[k] for k in keys if k in row})
        mlflow.log_params({f'version_{k}':v for k,v in versions.items()})
        mlflow.log_params({f'config_{k}':str(v) for k,v in config.to_dict().items()})
        mlflow.log_metrics({k:float(v) for k,v in row.items() if isinstance(v,(int,float,np.number)) and np.isfinite(v)})
        return run.info.run_id


In [ ]:
%%writefile tests/conftest.py
"""ให้ pytest หาโมดูลใน src ได้โดยไม่ต้องติดตั้งโปรเจกต์"""
import sys
import os
from pathlib import Path
if sys.platform == "darwin":
    os.environ["OMP_NUM_THREADS"] = "1"
ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))


In [ ]:
%%writefile tests/test_hmda.py
"""ทดสอบเส้นทางรั่วและพฤติกรรม HMDA ที่มีผลต่อข้อสรุป"""
import json
from pathlib import Path
import numpy as np
import pandas as pd
import pytest
from config import Config
from fairgroups.hmda import (LeakageGuard,HMDAPreprocessor,POST,FEATURES,prohibited,
                             demographics,split_hmda,dti_numeric)
from fairgroups.hmda_groups import ClassClusters,fit_hbac
from fairgroups.run import choose_jtt,oof_errors
from fairgroups.models import predict_mlp
from fairgroups.metrics import evaluate

ROOT=Path(__file__).resolve().parents[1]


def test_all_post_decision_and_sensitive_features_rejected():
    y=np.tile([0,1],40)
    cols=FEATURES+list(POST)+['denial_reason-1','tract_minority_population_percent',
        'applicant_race-1','co-applicant_ethnicity-1','applicant_sex_observed',
        'applicant_age_above_62','derived_race','derived_ethnicity','derived_sex']
    x=pd.DataFrame({c:np.ones(80) for c in cols})
    guard=LeakageGuard().fit(x,y)
    assert set(guard.features_)==set(FEATURES)
    assert all(not prohibited(c) for c in guard.features_)
    for c in cols:
        if prohibited(c):
            with pytest.raises(AssertionError): HMDAPreprocessor().fit(pd.DataFrame({c:np.ones(80)}))


def test_missing_gap_train_only_and_dti_order():
    y=np.tile([0,1],40)
    x=pd.DataFrame({'loan_amount':np.arange(80,dtype=float),'income':np.where(y==0,'Exempt','40')})
    guard=LeakageGuard().fit(x,y)
    assert guard.features_==['loan_amount']
    assert next(r for r in guard.audit_ if r['column']=='income')['missing_gap']==1
    before=guard.features_.copy();guard.transform(x.assign(income='99',loan_amount=np.nan))
    assert guard.features_==before
    np.testing.assert_array_equal(dti_numeric(pd.Series(['<20%','20%-<30%','30%-<36%','36','50%-60%','>60%'])),[10,25,33,36,55,65])


def test_unknown_retained_but_excluded_from_fairness():
    frame=pd.DataFrame({'derived_race':['White','Race Not Available','Race Not Available','Black or African American'],
                         'derived_ethnicity':['Not Hispanic or Latino','Hispanic or Latino','Ethnicity Not Available','Not Hispanic or Latino'],
                         'derived_sex':['Male','Female','Sex Not Available','Joint'],'applicant_age':['35-44','<25','8888','>74']})
    s=demographics(frame)
    assert s.race_eth.tolist()==['NH White','Hispanic','Unknown','NH Black']
    assert len(s)==4
    m=evaluate(np.array([1,0,0,1]),np.array([.9,.1,.8,.8]),s,min_detail=1)
    assert m['excluded_unknown_race_eth']==1 and m['excluded_unknown_sex']==1
    assert 'Unknown' not in json.loads(m['group_accuracy_intersection'])


def test_hmda_unsupervised_fits_never_include_holdout():
    rng=np.random.default_rng(17);n=160
    frame=pd.DataFrame({'loan_amount':rng.normal(size=n),'income':rng.normal(size=n),'lei':np.tile(['a','b','c','d'],40)},index=np.arange(500,500+n))
    y=np.tile([0,1],80)
    s=pd.DataFrame({'race_eth':np.tile(['NH White','NH Black'],80),'sex':'Male','age':'35-44','intersection':np.tile(['a','b'],80)},index=frame.index)
    splits=split_hmda(frame,y,s,17)
    tr=splits['train'];guard=LeakageGuard().fit(tr['frame'],tr['y']);tr['x']=guard.transform(tr['frame'])
    pre=HMDAPreprocessor(2,lei_top_n=1).fit(tr['x']);x=pre.transform(tr['x'])
    km=ClassClusters().fit(x,tr['y'],tr['ids'],2,17)
    saved=km.centers_.copy();pre.transform(guard.transform(splits['test']['frame'].assign(income=99999)))
    km.predict(x*100);np.testing.assert_array_equal(km.centers_,saved)
    for obj in [guard,pre,km]:
        assert set(obj.fit_ids_)==set(tr['ids'])
        assert set(obj.fit_ids_).isdisjoint(np.r_[splits['test']['ids'],splits['val']['ids']])
    error,audit=oof_errors(tr,Config(oof_folds=2,trees=3),17)
    held=[]
    for f in audit['folds']:
        assert set(f['fit_ids']).isdisjoint(f['heldout_ids'])
        assert set(f['guard_fit_ids'])==set(f['preprocessor_fit_ids'])==set(f['fit_ids'])
        held.extend(f['heldout_ids'])
    assert sorted(held)==sorted(tr['ids']) and np.isfinite(error).all()


def test_hbac_package_assigns_new_rows_without_bias_metric():
    rng=np.random.default_rng(2);x=np.r_[rng.normal(-3,.1,(40,3)),rng.normal(3,.1,(40,3))].astype(np.float32)
    m=fit_hbac(x,np.r_[np.zeros(40),np.ones(40)],np.arange(80),17,k=2)
    assert m.n_clusters_==2
    np.testing.assert_array_equal(m.predict(x),m.labels_)


def test_jtt_validation_grid_does_not_collapse_on_separable_data():
    rng=np.random.default_rng(7);x=rng.normal(size=(400,4)).astype(np.float32);y=(x[:,0]>0).astype(int)
    cfg=Config(epochs=12,jtt_identification_epochs=3,hidden=16,batch_size=32)
    cp,weight,history=choose_jtt(x[:300],{'y':y[:300]},x[300:],{'y':y[300:]},np.zeros(100),cfg,17)
    p=predict_mlp(cp['average'][0],x[300:]);assert .05<(p>=.5).mean()<.95
    assert weight==max(history,key=lambda r:r['validation_accuracy'])['upweight']


def test_hmda_recorded_audit_and_jtt():
    path=ROOT/'results/leakage_audit.json'
    if not path.exists(): pytest.skip('ยังไม่มีผล HMDA จริง')
    audits=json.loads(path.read_text())['seeds']
    for a in audits:
        train,val,test=(set(a[f'{n}_ids']) for n in ['train','val','test'])
        assert train.isdisjoint(val|test) and val.isdisjoint(test)
        assert all(not prohibited(c) for c in a['features'])
        for f in a['fits']:
            assert set(f['fit_ids']).issubset(train) and set(f['fit_ids']).isdisjoint(val|test)
        for oof in a['error_crossfits']:
            held=[]
            for f in oof['folds']:
                assert set(f['fit_ids']).isdisjoint(f['heldout_ids'])
                assert set(f['fit_ids'])|set(f['heldout_ids'])==train
                assert set(f['guard_fit_ids'])==set(f['preprocessor_fit_ids'])==set(f['fit_ids'])
                held.extend(f['heldout_ids'])
            assert sorted(held)==sorted(train)
        for r in a['column_audit']:
            if not r['dropped']: assert r['missing_gap']<=.10
    results=pd.read_csv(ROOT/'results/hmda_pilot_results.csv')
    jtt=results[(results.method=='JTT') & (results.selection=='average')]
    # ถ้าโมเดลยุบเป็นคลาสเดียว งานจริงต้อง fail ให้แก้หรือรายงาน ไม่ซ่อนด้วย EO
    assert jtt.positive_rate.between(.005,.995,inclusive='neither').all()


def test_demo_build_and_prediction():
    if not (ROOT/'results/demo_bundle.joblib').exists(): pytest.skip('ยังไม่มี bundle จาก pilot')
    from fairgroups.demo import load_bundle,predict_application,build_interface
    b=load_bundle(ROOT/'results');values=[b['defaults'][c] for c in b['inputs']]
    result,card=predict_application(b,*values)
    assert 0<=result['โอกาสอนุมัติ ERM']<=1
    assert 0<=result['โอกาสอนุมัติหลังฝึก DRO']<=1
    assert 'วิจัย' in card and str(result['กลุ่ม']) in card
    assert len(b['inputs'])==8
    demo=build_interface(ROOT/'results');assert callable(demo.fn)
    second=demo.fn(*values);assert second[0]==result


In [ ]:
%%writefile tests/test_leakage.py
"""ตรวจการรั่วจากเลขแถวจริงและจากข้อมูลจำลองที่มีค่า holdout แปลกออกไป"""
import json
from pathlib import Path
import numpy as np
import pandas as pd
import pytest
from config import Config
from fairgroups.data import (TrainPreprocessor, FEATURES, NUMERIC, SENSITIVE,
                             split_data, labeled_indices)
from fairgroups.representations import crossfit_error, train_scarf
from fairgroups.clustering import discover


def sample_data():
    rng = np.random.default_rng(12)
    n = 160
    x = pd.DataFrame({c: rng.integers(1, 6, n).astype(float) for c in FEATURES},
                     index=np.arange(1000, 1000+n))
    y = np.tile([0, 1], n // 2)
    s = pd.DataFrame({"SEX": np.tile([1, 1, 2, 2], n//4),
                      "RAC1P": np.tile([1, 1, 1, 1, 2, 2, 2, 2], n//8)}, index=x.index)
    return x, y, s


def test_train_only_scaler_encoder_scarf_kmeans():
    x, y, s = sample_data()
    split = split_data(x, y, s, seed=17)
    train, test = split["train"], split["test"]
    processor = TrainPreprocessor(2).fit(train["x"])
    test_changed = test["x"].copy()
    test_changed.loc[:, "AGEP"] = 99999
    processor.transform(test_changed)
    assert not SENSITIVE.intersection(processor.feature_names_in_)
    assert set(processor.fit_ids_).isdisjoint(test["ids"])
    expected_mean = train["x"][NUMERIC].mean().to_numpy()
    np.testing.assert_allclose(processor.encoder.named_transformers_["numeric"].mean_, expected_mean)
    config = Config(scarf_epochs=1, hidden=8, embedding=4, batch_size=32)
    scarf = train_scarf(train["x"], processor, config, 17)
    _, _, km = discover(processor.transform(train["x"]), train["ids"], train["s"], 2, 17)
    for estimator in [processor, scarf, km]:
        assert set(estimator.fit_ids_) == set(train["ids"])
        assert set(estimator.fit_ids_).isdisjoint(test["ids"])
        assert set(estimator.fit_ids_).isdisjoint(split["val"]["ids"])


def test_sensitive_features_are_rejected():
    x, _, _ = sample_data()
    for sensitive in SENSITIVE:
        with pytest.raises(AssertionError):
            TrainPreprocessor().fit(x.assign(**{sensitive: 1}))


def test_oof_exactly_once_within_label_budget():
    x, y, s = sample_data()
    part = split_data(x, y, s, seed=17)["train"]
    labeled = labeled_indices(part["y"], 0.5, 17)
    processor = TrainPreprocessor(2).fit(part["x"])
    ids = part["ids"][labeled]
    rep, audit = crossfit_error(processor.transform(part["x"])[labeled], part["y"][labeled],
                               ids, Config(trees=3), 17)
    assert rep.shape == (len(labeled), 3)
    held = []
    for fold in audit["folds"]:
        assert set(fold["fit_ids"]).isdisjoint(fold["heldout_ids"])
        assert set(fold["fit_ids"]) | set(fold["heldout_ids"]) == set(ids)
        held.extend(fold["heldout_ids"])
    assert sorted(held) == sorted(ids)
    assert len(ids) == len(part["y"]) // 2


def test_recorded_experiment_has_no_holdout_in_fit():
    path = Path(__file__).resolve().parents[1] / "results/acs_round1/leakage_audit.json"
    if not path.exists():
        pytest.skip("ยังไม่ได้รันชุดจริง; หลัง --quick จะตรวจหลักฐานนี้อีกครั้ง")
    for audit in json.loads(path.read_text()):
        train, val, test = (set(audit[f"{name}_ids"]) for name in ["train", "val", "test"])
        assert train.isdisjoint(val) and train.isdisjoint(test) and val.isdisjoint(test)
        assert not SENSITIVE.intersection(audit["features"])
        for fit in audit["fits"]:
            assert set(fit["fit_ids"]).issubset(train)
            assert set(fit["fit_ids"]).isdisjoint(test | val)
        for error in audit["error_crossfits"]:
            labeled = set(error["fit_ids"])
            assert len(labeled) == int(len(train) * error["label_fraction"])
            held = []
            for fold in error["folds"]:
                assert set(fold["fit_ids"]).isdisjoint(fold["heldout_ids"])
                assert set(fold["fit_ids"]) | set(fold["heldout_ids"]) == labeled
                held.extend(fold["heldout_ids"])
            assert len(held) == len(set(held)) == len(labeled)


In [ ]:
%%writefile tests/test_methods.py
"""ตรวจสูตรที่มีผลต่อข้อสรุป ไม่ใช่เพียงตรวจว่าโค้ดเรียกได้"""
import numpy as np
import torch
from config import Config
from fairgroups.dro import group_dro_loss
from fairgroups.metrics import group_metrics
from fairgroups.models import train_mlp, predict_mlp
from fairgroups.efficiency import network_cost


def test_dro_averages_groups_not_examples():
    losses = torch.tensor([1., 1., 4.], requires_grad=True)
    groups = torch.tensor([0, 0, 1])
    value = group_dro_loss(losses, groups, torch.zeros(2), eta=0)
    assert float(value.detach()) == 2.5
    value.backward()
    np.testing.assert_allclose(losses.grad.numpy(), [0.25, 0.25, 0.5])


def test_known_fairness_values():
    y = np.array([1, 1, 0, 0, 1, 1, 0, 0])
    pred = np.array([1, 1, 0, 0, 1, 0, 1, 0])
    scores = group_metrics(y, pred, np.array(["a"]*4 + ["b"]*4))
    assert scores["eo_gap"] == 0.5
    assert scores["dp_difference"] == 0
    assert scores["worst_group_accuracy"] == 0.5


def test_average_selection_is_invariant_to_population_groups():
    rng = np.random.default_rng(1)
    x = rng.normal(size=(40, 4)).astype(np.float32)
    y = (x[:, 0] > 0).astype(int)
    cfg = Config(epochs=3, hidden=8, batch_size=16)
    a, _ = train_mlp(x, y, x, y, np.zeros(40), cfg, seed=17)
    b, _ = train_mlp(x, y, x, y, np.arange(40) % 4, cfg, seed=17)
    assert a["average"][1] == b["average"][1]
    np.testing.assert_array_equal(predict_mlp(a["average"][0], x), predict_mlp(b["average"][0], x))


def test_linear_flops_formula():
    model = torch.nn.Sequential(torch.nn.Linear(3, 4), torch.nn.ReLU(), torch.nn.Linear(4, 1))
    assert network_cost(model) == {"parameters": 21, "flops_per_row": 32}


In [ ]:
%%writefile tests/test_tracking.py
"""SQLite ต้องอ่านผลกลับได้ และไม่มี artifact รายรัน"""
import mlflow
from fairgroups.tracking import setup_tracking, log_row
from config import Config


def test_sqlite_tracking_has_no_per_run_artifacts(tmp_path):
    setup_tracking(tmp_path)
    run_id=log_row({'seed':17,'representation':'raw','method':'test','accuracy':.75},{},Config())
    fetched=mlflow.get_run(run_id)
    assert fetched.data.metrics['accuracy']==.75
    assert fetched.data.params['seed']=='17'
    assert fetched.info.status=='FINISHED'
    assert mlflow.get_tracking_uri().startswith('sqlite:///')
    assert not mlflow.tracking.MlflowClient().list_artifacts(run_id)
    assert len([p for p in tmp_path.rglob('*') if p.is_file() and not p.name.startswith('._')])<=3


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pytest', 'tests', '-q'])


In [ ]:
import time
start = time.perf_counter()
subprocess.check_call([sys.executable, 'run_pilot.py', '--quick' if QUICK else '--full'])
print(f'Wall time: {time.perf_counter()-start:.1f} seconds')
subprocess.check_call([sys.executable, '-m', 'pytest', 'tests', '-q'])


In [ ]:
from IPython.display import display, Markdown, Image
display(Markdown(Path('results/hmda_pilot_summary.md').read_text()))
for path in sorted(Path('results/figures').glob('*.png')):
    display(Image(filename=str(path)))


In [ ]:
# ดาวน์โหลดผลรวม (ไม่รวม data และ mlruns ที่มีขนาดใหญ่)
import shutil
shutil.make_archive('pilot_results', 'zip', 'results')
try:
    from google.colab import files
    files.download('pilot_results.zip')
except ImportError:
    print(Path('pilot_results.zip').resolve())


In [ ]:
# หน้าสาธิตท้าย notebook: ต้นแบบเพื่อการวิจัยเท่านั้น
import sys
sys.path.insert(0, str(BASE / 'src'))
from fairgroups.demo import build_interface
demo = build_interface(BASE / 'results')
demo.launch(share=True)
